# Algorithm24 Tutorial Audit — EM-Guided q-Region CPS for KLDM

This notebook is a lightweight, laptop-friendly feasibility notebook for Algorithm24. It treats a cheap facitKLDM EM lookahead endpoint as a noisy structural observation, explains it under oracle-space-group Wyckoff templates, builds a small q-region between the current KLDM explanation and the EM explanation, and tests whether that region yields safer or better CPS interventions.

The main questions are:
- is the EM endpoint explainable under the oracle space group?
- does the q-region between `q_now` and `q_EM` contain better candidates than `q_now` alone?
- does CrystalFormer help rank candidates inside that small filtered region?
- does a weak one-step CPS intervention selected from that region survive KLDM better than the simpler Algorithm22 geometry baseline?


## CrystalFormer weights

Recommended reference links:
- [CrystalFormer available weights](https://github.com/deepmodeling/CrystalFormer#available-weights)
- [CrystalFormer CSP Colab](https://colab.research.google.com/drive/1I7b5exbB2oBjexFIEaeDQexmYRDgLHVk?authuser=0#scrollTo=kfu6Ez9e6Sp7)

The CSP-only checkpoint expected by this notebook is:
- `epoch_046000.pkl`

You can either:
- set `KLDM_ALGO21_CF_CHECKPOINT` to a local checkpoint path
- or run the download cell below


## CrystalFormer dependency bootstrap

**Purpose:** CrystalFormer needs a small JAX-side stack before the checkpoint wrapper can load. If the preflight below reports missing packages such as `haiku`, run the optional install cell once, restart the kernel, and rerun from the top.


In [1]:
# Optional dependency bootstrap for CrystalFormer.
# Run this only if the dependency preflight reports missing packages.

# %pip install -U "jax[cpu]"
# %pip install dm-haiku==0.0.15 optax==0.2.6 pyxtal==1.1.1 gdown


In [2]:
import importlib.util
import pandas as pd
rows = []
for module_name in ('jax', 'haiku', 'optax', 'pyxtal', 'gdown'):
    rows.append({
        'test': 'algorithm22_cf_dependency_preflight',
        'module': module_name,
        'available': bool(importlib.util.find_spec(module_name) is not None),
    })

dep_df = pd.DataFrame(rows)
display(dep_df)


,test,module,available
0,algorithm22_cf_dependency_preflight,jax,True
1,algorithm22_cf_dependency_preflight,haiku,True
2,algorithm22_cf_dependency_preflight,optax,True
3,algorithm22_cf_dependency_preflight,pyxtal,True
4,algorithm22_cf_dependency_preflight,gdown,True


In [3]:
import os
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_ALLOCATOR", "platform")
os.environ.setdefault("JAX_PLATFORM_NAME", "cpu")
os.environ["JAX_DISABLE_JIT"] = str(os.environ.get("KLDM_ALGO23_DISABLE_JIT", "false"))
os.environ["KLDM_ALGO21_SAFE_MODE"] = str(os.environ.get("KLDM_ALGO23_CF_SAFE_MODE", "false"))
os.environ.setdefault("KLDM_ALGO23_ENABLE_CF_KERNEL_FALLBACK", "false")
os.environ.setdefault("KLDM_ALGO23_CF_RUNTIME_MODE", "subprocess_primary")
import time
import math
import traceback
import subprocess
import sys
import pickle
import json as jsonlib
import importlib
import gc
from dataclasses import dataclass, replace
from pathlib import Path
from types import SimpleNamespace
from typing import Any

import numpy as np
import pandas as pd
import torch
import yaml
from pymatgen.core import Element

ROOT = Path.cwd().resolve()
if not ((ROOT / 'configs').exists() and (ROOT / 'src').exists()):
    if ((ROOT.parent / 'configs').exists() and (ROOT.parent / 'src').exists()):
        ROOT = ROOT.parent

import kldmPlus.symmetry.crystalformer_backend as cf_backend
import kldmPlus.algorithm22_faithful_kldm_cps_csp as algo22_backend
import kldmPlus.algorithm24_em_guided_q_region_cps as algo24_backend
cf_backend = importlib.reload(cf_backend)
algo22_backend = importlib.reload(algo22_backend)
algo24_backend = importlib.reload(algo24_backend)

from kldmPlus.run_sampling_compare import SamplingCompareRunner, set_seed
from kldmPlus.sample_evaluation import build_structure_from_sample, evaluate_csp_reconstruction
from kldmPlus.symmetry import build_symmetry_frame_bridge, estimate_semantic_transport_for_reference_order, oracle_spacegroup_from_case
from kldmPlus.utils.time import iter_sampling_times, make_times
from kldmPlus.symmetry.wyckoff_templates import extract_wyckoff_templates, flatten_site_signature, sample_random_free_vars, recover_template_free_vars_from_anchor_entries

Algorithm19State = algo22_backend.Algorithm19State
build_oracle_diffcsppp_payload_from_structure = algo22_backend.build_oracle_diffcsppp_payload_from_structure
map_model_to_payload_reference_chart = algo22_backend.map_model_to_payload_reference_chart
map_payload_reference_chart_to_model_frame = algo22_backend.map_payload_reference_chart_to_model_frame
kldm_clean_fractional_denoiser_Df = algo22_backend.kldm_clean_fractional_denoiser_Df
wrap01 = algo22_backend.wrap01
wrapdiff = algo22_backend.wrapdiff
torus_rmse = algo22_backend.torus_rmse

CrystalFormerLikelihood = algo22_backend.CrystalFormerLikelihood
predict_clean_f0 = algo22_backend.predict_clean_f0
model_to_payload = algo22_backend.model_to_payload
payload_to_model = algo22_backend.payload_to_model
expand_q = algo22_backend.expand_q
fit_q_to_clean_prediction = algo22_backend.fit_q_to_clean_prediction
torus_soft_project = algo22_backend.torus_soft_project
renoise_from_f0 = algo22_backend.renoise_from_f0
sample_q_from_crystalformer = algo22_backend.sample_q_from_crystalformer
build_payload_from_template_q = algo22_backend.build_payload_from_template_q
species_match_reorder = algo22_backend.species_match_reorder
Algorithm21Config = cf_backend.Algorithm21Config
q_only_clean_cf_fit = cf_backend.q_only_clean_cf_fit
q_only_clean_cf_local_rerank = cf_backend.q_only_clean_cf_local_rerank
torus_interp = cf_backend.torus_interp
rank_q_candidates = getattr(cf_backend, 'rank_q_candidates', None)
build_crystalformer_reduced_sequence = getattr(cf_backend, 'build_crystalformer_reduced_sequence', None)
expand_crystalformer_reduced_sequence = getattr(cf_backend, 'expand_crystalformer_reduced_sequence', None)
crystalformer_reduced_sequence_debug = getattr(cf_backend, 'crystalformer_reduced_sequence_debug', None)
crystalformer_site_representative_search = getattr(cf_backend, 'crystalformer_site_representative_search', None)
crystalformer_payload_order_assembly_debug = getattr(cf_backend, 'crystalformer_payload_order_assembly_debug', None)

CONFIG_PATH = ROOT / 'configs' / 'kldm_plus' / 'mp_20' / 'mp20_sampling_compare_em_vs_algorithm10_casal_chart.yaml'
with CONFIG_PATH.open('r', encoding='utf-8') as handle:
    CONFIG = yaml.safe_load(handle)

PROFILE = os.environ.get('KLDM_ALGO21_PROFILE', 'laptop').strip().lower()
def profile_default(name: str, laptop: str, deep: str | None = None) -> str:
    if name in os.environ:
        return os.environ[name]
    if PROFILE in {'full', 'deep'} and deep is not None:
        return deep
    return laptop

def notebook_log(message: str) -> None:
    text = str(message)
    print(text)
    with ALGO21_LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(text)
        if not text.endswith("\n"):
            handle.write("\n")

def cf_nll_eval(*, payload, q, lattice_feature, formula, label: str) -> float:
    notebook_log(f"[cf-eval] start {label}")
    if not CF_READY:
        notebook_log(f"[cf-eval] skip {label} CF unavailable")
        return float('nan')
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    try:
        q_np = np.asarray(torch.as_tensor(q).detach().cpu(), dtype=float)
        l_np = np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float)
        if use_subprocess:
            tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
            tmp_dir.mkdir(parents=True, exist_ok=True)
            safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
            in_path = tmp_dir / f'{safe_label}.pkl'
            out_path = tmp_dir / f'{safe_label}.json'
            with in_path.open('wb') as handle:
                pickle.dump({
                    'repo_root': str(ROOT),
                    'checkpoint_path': str(CF_CHECKPOINT),
                    'symmetry_payload': payload,
                    'q': q_np,
                    'lattice_feature': l_np,
                    'formula': formula,
                }, handle)
            cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_once.py'), '--input', str(in_path), '--output', str(out_path)]
            notebook_log(f"[cf-eval] subprocess start {label}")
            completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
            result = {}
            if out_path.exists():
                with out_path.open('r', encoding='utf-8') as handle:
                    result = jsonlib.load(handle)
            if completed.returncode != 0 or not result.get('ok'):
                notebook_log(f"[cf-eval] subprocess ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
                raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer subprocess failed')
            value = float(result['value'])
        else:
            raise RuntimeError('In-kernel CrystalFormer evaluation is disabled for Algorithm22 notebook stability.')
        notebook_log(f"[cf-eval] done {label} value={value:.6g}")
        return value
    except Exception as exc:
        notebook_log(f"[cf-eval] ERROR {label} {type(exc).__name__}: {exc}")
        raise
    finally:
        notebook_cleanup()

def cf_sequence_smoke(*, payload, q, lattice_feature, label: str) -> dict[str, Any]:
    notebook_log(f"[cf-seq] start {label}")
    seq = build_crystalformer_reduced_sequence(payload=payload, q=q, lattice_feature=lattice_feature)
    notebook_log(f"[cf-seq] done {label} num_sites={int(seq['num_sites'])}")
    notebook_cleanup()
    return seq

def parse_int_list(text: str) -> list[int]:
    return [int(v.strip()) for v in str(text).split(',') if v.strip()]

def parse_float_list(text: str) -> list[float]:
    return [float(v.strip()) for v in str(text).split(',') if v.strip()]

SAMPLE_SEED = int(profile_default('KLDM_ALGO21_SEED', '2', '2'))
GRAPH_IDS = parse_int_list(profile_default('KLDM_ALGO21_GRAPH_IDS', '2', '2'))
CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_CORE_T', '0.1,0.3,0.5', '0.1,0.3,0.5')))
BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_BETA_VALUES', '0,0.001,0.01,0.1,1,10', '0,0.001,0.01,0.1,1,10')))
ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_ALPHA_VALUES', '0,0.1,0.25,0.5,1.0', '0,0.1,0.25,0.5,1.0')))
Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_Q_OPT_STEPS', '50', '50'))
TEST45_BETA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_BETA_VALUES', '0', '0')))
TEST6_ALPHA_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST6_ALPHA_VALUES', '0,0.25', '0,0.25')))
TEST46_Q_OPT_STEPS = int(profile_default('KLDM_ALGO21_TEST46_Q_OPT_STEPS', '4', '4'))
TEST46_CORE_T_VALUES = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST46_CORE_T', '0.5', '0.5')))
TEST7_RERANK_RADIUS = float(profile_default('KLDM_ALGO21_TEST7_RERANK_RADIUS', '0.05', '0.05'))
TEST7_RERANK_CANDIDATES = int(profile_default('KLDM_ALGO21_TEST7_RERANK_CANDIDATES', '3', '3'))
TEST7_RERANK_KEEP_TOL = float(profile_default('KLDM_ALGO21_TEST7_RERANK_KEEP_TOL', '0.05', '0.05'))
TEST45_CF_SCALE_BETAS = tuple(parse_float_list(profile_default('KLDM_ALGO21_TEST45_CF_SCALE_BETAS', '0,1e-6,1e-5,1e-4', '0,1e-6,1e-5,1e-4')))
# CF-gradient optimization is deliberately opt-in; default notebook tests use value-only reranking.
os.environ.setdefault('KLDM_ALGO21_ENABLE_CF_GRAD', 'false')
SHORT_SAMPLER_STEPS = int(profile_default('KLDM_ALGO21_SHORT_SAMPLER_STEPS', '100', '100'))
SHORT_SAMPLER_TSTART = float(profile_default('KLDM_ALGO21_SHORT_SAMPLER_TSTART', '0.5', '0.5'))
NOTEBOOK_SAFE_MODE = str(os.environ.get("KLDM_ALGO21_SAFE_MODE", "true")).strip().lower() in {"1", "true", "yes", "on"}
SHOW_FULL_DEBUG_TABLES = not NOTEBOOK_SAFE_MODE
CF_RANDOM_TRIALS = int(profile_default('KLDM_ALGO21_CF_RANDOM_TRIALS', '1' if NOTEBOOK_SAFE_MODE else '8', '1'))
CF_CHECKPOINT = str(profile_default('KLDM_ALGO21_CF_CHECKPOINT', str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl')), str((ROOT / 'notebooks' / 'epoch_046000.pkl') if (ROOT / 'notebooks' / 'epoch_046000.pkl').exists() else (ROOT / 'epoch_046000.pkl'))))
CF_FORMULA = os.environ.get('KLDM_ALGO21_FORMULA', '').strip() or None
ALGO21_LOG_PATH = ROOT / 'notebooks' / 'log.txt'
os.environ["KLDM_ALGO21_LOG_PATH"] = str(ALGO21_LOG_PATH)
ALGO21_LOG_PATH.write_text("", encoding="utf-8")

runner = SamplingCompareRunner(config_path=CONFIG_PATH)
runner.model.eval()
set_seed(SAMPLE_SEED)

requested_num_targets = max(max(GRAPH_IDS) if GRAPH_IDS else 0, len(GRAPH_IDS), 5)
if int(runner.compare_cfg.get('num_targets', 0)) < requested_num_targets:
    runner.compare_cfg['num_targets'] = int(requested_num_targets)
    runner.compare_cfg['batch_size'] = max(int(runner.compare_cfg.get('batch_size', 0)), int(requested_num_targets))
    runner.loader, runner.lattice_transform = runner._build_loader()
full_batch = next(iter(runner.loader)).to(runner.device)
full_ptr = full_batch.ptr.tolist()
full_num_graphs = max(len(full_ptr) - 1, 0)
available_graph_ids = [int(graph_id) for graph_id in GRAPH_IDS if 1 <= int(graph_id) <= full_num_graphs]
selected_graph_indices0 = [int(graph_id) - 1 for graph_id in available_graph_ids]
selected_items = full_batch.index_select(selected_graph_indices0) if hasattr(full_batch, 'index_select') else full_batch[selected_graph_indices0]
if isinstance(selected_items, list):
    batch = full_batch.__class__.from_data_list(selected_items)
else:
    batch = selected_items
batch = batch.to(runner.device)
ptr = batch.ptr.detach().cpu().tolist()

print(f'profile={PROFILE} graphs={GRAPH_IDS} t={CORE_T_VALUES} beta={BETA_VALUES} alpha={ALPHA_VALUES} q_opt_steps={Q_OPT_STEPS} cf_ckpt={CF_CHECKPOINT}')


def notebook_cleanup():
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        try:
            cf_obj = globals().get('cf_like', None)
            if cf_obj is not None and hasattr(cf_obj, 'release_runtime'):
                cf_obj.release_runtime()
        except Exception:
            pass
    try:
        worker = globals().get('CF_WORKER_PROC', None)
        if worker is not None:
            try:
                if worker.stdin is not None:
                    worker.stdin.write(jsonlib.dumps({'cmd': 'shutdown'}) + '\n')
                    worker.stdin.flush()
            except Exception:
                pass
            try:
                worker.wait(timeout=5)
            except Exception:
                try:
                    worker.terminate()
                except Exception:
                    pass
        globals()['CF_WORKER_PROC'] = None
    except Exception:
        pass
    gc.collect()
    try:
        import jax
        jax.clear_caches()
    except Exception:
        pass
    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception:
        pass

Algorithm22Config = algo22_backend.Algorithm22Config
algorithm22_projection_schedule = algo22_backend.algorithm22_projection_schedule
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_cps_gamma = algo22_backend.algorithm22_cps_gamma
algorithm22_cps_alpha = algo22_backend.algorithm22_cps_alpha
clean_fractional_estimate_22 = algo22_backend.clean_fractional_estimate
clean_lattice_estimate_22 = algo22_backend.clean_lattice_estimate
decode_state_cell_matrix_22 = algo22_backend.decode_state_cell_matrix
oracle_template_projection_22 = algo22_backend.oracle_template_projection
generate_candidates_22 = algo22_backend.generate_candidates
generate_pyxtal_candidates_22 = algo22_backend.generate_pyxtal_candidates
rank_candidates_against_clean_estimate_22 = algo22_backend.rank_candidates_against_clean_estimate
algorithm22b_ranked_kldm_cps_step = algo22_backend.algorithm22b_ranked_kldm_cps_step
group_witness_from_candidates_22 = algo22_backend.group_witness_from_candidates


Algorithm22RunResult = algo22_backend.Algorithm22RunResult
algorithm22A_oracle_template_kldm_cps = algo22_backend.algorithm22A_oracle_template_kldm_cps
algorithm22B_ranked_kldm_cps = algo22_backend.algorithm22B_ranked_kldm_cps
kldm_clean_fractional_denoiser_22 = algo22_backend.kldm_clean_fractional_denoiser
kldm_clean_lattice_denoiser_22 = algo22_backend.kldm_clean_lattice_denoiser
expand_template_to_model_order_22 = algo22_backend.expand_template_to_model_order
materialize_candidate_from_template_22 = algo22_backend.materialize_candidate_from_template
rank_projected_candidates_with_rule_22 = algo22_backend.rank_projected_candidates_with_rule
sample_template_q_proposals_22 = algo22_backend.sample_template_q_proposals
safety_ok_22 = algo22_backend.safety_ok
torus_distance_squared_22 = algo22_backend.torus_distance_squared
oracle_template_witness_22 = algo22_backend.oracle_template_witness
project_candidates_to_clean_estimate_22 = algo22_backend.project_candidates_to_clean_estimate
algorithm22_piecewise_alpha = algo22_backend.algorithm22_piecewise_alpha
algorithm22_piecewise_beta = algo22_backend.algorithm22_piecewise_beta
kldm_cps_update_or_renoise_22 = algo22_backend.kldm_cps_update_or_renoise
tdm_velocity_sigma_at_state_22 = algo22_backend.tdm_velocity_sigma_at_state
tdm_residual_epsilon_r_22 = algo22_backend.tdm_residual_epsilon_r


MODELS_PROJECT_ROOT: /workspace/.venv/lib/python3.11/site-packages/mattergen
mps:  False
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
dataset_cache action=load dataset=mp_20 split=test reason=fresh path=/workspace/data/mp_20/processed/test
dataset_cache action=from_cache_path:start dataset=mp_20 split=test
dataset_cache action=from_cache_path:done dataset=mp_20 split=test
dataset_cache action=builder_build:start dataset=mp_20 split=test
dataset_cache action=builder_build:done dataset=mp_20 split=test
profile=laptop graphs=[2] t=(0.1, 0.3, 0.5) beta=(0.0, 0.001, 0.01, 0.1, 1.0, 10.0) alpha=(0.0, 0.1, 0.25, 0.5, 1.0) q_opt_steps=50 cf_ckpt=/workspace/notebooks/epoch_

In [4]:
@dataclass
class GraphCase:
    graph_id: int
    graph_idx0: int
    composition: dict[int, int]
    atomic_numbers: torch.Tensor
    gt_frac: torch.Tensor
    gt_l_feature: torch.Tensor
    requested_sg: int

result_tables: dict[str, pd.DataFrame] = {}
payload_cache: dict[int, Any] = {}


def dataframe_result(name: str, rows: list[dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    if 'PASS' not in df.columns:
        df['PASS'] = False
    if 'status' not in df.columns:
        df['status'] = np.where(df['PASS'], 'PASS', 'FAIL')
    result_tables[name] = df
    return df


def safe_display_sorted(df: pd.DataFrame, by: list[str]):
    df = df.copy()
    cols = list(df.columns)
    for key in by:
        if key not in cols:
            df[key] = np.nan
    display(df.sort_values(by).reset_index(drop=True))


def error_row(test: str, graph: int | str, exc: Exception, failure_category: str, **extra) -> dict[str, Any]:
    tb = traceback.format_exc().strip().splitlines()
    return {
        'test': test,
        'graph': graph,
        'PASS': False,
        'status': 'ERROR',
        'error_type': type(exc).__name__,
        'error_message': str(exc),
        'traceback_tail': tb[-1] if tb else '',
        'failure_category': failure_category,
        **extra,
    }


def graph_slice(graph_idx0: int) -> tuple[int, int]:
    return int(ptr[graph_idx0]), int(ptr[graph_idx0 + 1])


def composition_counter(values) -> dict[int, int]:
    arr = [int(v) for v in torch.as_tensor(values, dtype=torch.long).reshape(-1).tolist()]
    from collections import Counter
    return dict(sorted(Counter(arr).items()))


def graph_edge_node_index(case: GraphCase) -> torch.Tensor:
    start, end = graph_slice(case.graph_idx0)
    edge_index = batch.edge_node_index
    mask = ((edge_index[0] >= start) & (edge_index[0] < end) & (edge_index[1] >= start) & (edge_index[1] < end))
    return (edge_index[:, mask] - start).detach().clone()


def graph_tensors(graph_idx0: int) -> dict[str, Any]:
    start, end = graph_slice(graph_idx0)
    return {
        'pos': batch.pos[start:end].detach().clone(),
        'l': batch.l[graph_idx0].detach().clone().reshape(-1),
        'h': batch.atomic_numbers[start:end].detach().clone().to(torch.long),
        'sg': int(torch.as_tensor(batch.space_group).reshape(-1)[graph_idx0].item()),
    }


def load_test_graphs(graph_ids=available_graph_ids) -> list[GraphCase]:
    out = []
    for graph_idx0, graph_id in enumerate(graph_ids):
        g = graph_tensors(graph_idx0)
        out.append(GraphCase(
            graph_id=int(graph_id),
            graph_idx0=int(graph_idx0),
            composition=composition_counter(g['h']),
            atomic_numbers=g['h'],
            gt_frac=g['pos'],
            gt_l_feature=g['l'],
            requested_sg=g['sg'],
        ))
    return out

GRAPH_CASES = load_test_graphs(available_graph_ids)
print('loaded_graphs=', [g.graph_id for g in GRAPH_CASES], 'sg=', [g.requested_sg for g in GRAPH_CASES])


def make_single_graph_batch_view(case: GraphCase, *, ref_tensor: torch.Tensor | None = None):
    device = case.gt_frac.device if ref_tensor is None else ref_tensor.device
    dtype = case.gt_frac.dtype if ref_tensor is None else ref_tensor.dtype
    pos = case.gt_frac.detach().clone().to(device=device, dtype=dtype)
    l = case.gt_l_feature.detach().clone().reshape(1, -1).to(device=device, dtype=case.gt_l_feature.dtype if ref_tensor is None else case.gt_l_feature.dtype)
    atomic_numbers = case.atomic_numbers.detach().clone().to(device=device, dtype=torch.long)
    batch_index = torch.zeros(pos.shape[0], device=device, dtype=torch.long)
    num_atoms = torch.tensor([int(pos.shape[0])], device=device, dtype=torch.long)
    edge_node_index = graph_edge_node_index(case).to(device=device)
    space_group = torch.tensor([int(case.requested_sg)], device=device, dtype=torch.long)

    class _SingleGraphBatch(SimpleNamespace):
        def to(self, device):
            out = _SingleGraphBatch()
            for key, value in self.__dict__.items():
                if torch.is_tensor(value):
                    setattr(out, key, value.to(device))
                else:
                    setattr(out, key, value)
            return out

    return _SingleGraphBatch(
        pos=pos,
        l=l,
        atomic_numbers=atomic_numbers,
        batch=batch_index,
        num_atoms=num_atoms,
        num_graphs=1,
        edge_node_index=edge_node_index,
        space_group=space_group,
    )


def build_case_structure(case: GraphCase):
    return build_structure_from_sample(case.gt_frac, case.gt_l_feature, case.atomic_numbers, lattice_transform=runner.lattice_transform)


def build_oracle_payload(case: GraphCase):
    if case.graph_id in payload_cache:
        return payload_cache[case.graph_id]
    structure = build_case_structure(case)
    payload = build_oracle_diffcsppp_payload_from_structure(
        standardized_structure=structure,
        requested_spacegroup=oracle_spacegroup_from_case(case),
        tol=1e-2,
    )
    bridge = build_symmetry_frame_bridge(vanilla_structure=structure, standardization='refined', symprec=1e-2, angle_tolerance=5.0)
    tau_ref, assignment_ref, rmse_ref = estimate_semantic_transport_for_reference_order(
        standardized_reference_frac_coords=np.asarray(payload.expanded_frac_coords, dtype=float),
        standardized_reference_atomic_numbers=np.asarray(payload.expanded_atomic_numbers, dtype=int),
        bridge=bridge,
    )
    linear_std_to_model = np.asarray(bridge.standardized_to_vanilla_linear, dtype=float)
    linear_model_to_std = np.linalg.inv(linear_std_to_model)
    payload = replace(
        payload,
        debug_info={
            **(payload.debug_info or {}),
            'model_reference_frac_coords': np.asarray(structure.frac_coords, dtype=float).tolist(),
            'model_to_payload_linear': linear_model_to_std.tolist(),
            'model_to_payload_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'model_to_payload_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'payload_to_model_linear': linear_std_to_model.tolist(),
            'payload_to_model_tau': np.asarray(tau_ref, dtype=float).tolist(),
            'payload_to_model_order': np.asarray(assignment_ref, dtype=int).tolist(),
            'transport_rmse': float(rmse_ref),
        },
    )
    payload_cache[case.graph_id] = payload
    return payload


def evaluate_arrays(case: GraphCase, pred_f: torch.Tensor, pred_l: torch.Tensor, pred_h: torch.Tensor) -> dict[str, Any]:
    result = evaluate_csp_reconstruction(
        pred_f=pred_f,
        pred_l=pred_l,
        pred_a=pred_h.to(torch.long),
        target_f=case.gt_frac,
        target_l=case.gt_l_feature,
        target_a=case.atomic_numbers,
        lattice_transform=runner.lattice_transform,
        requested_space_group=case.requested_sg,
        validity_cutoff=0.5,
    )
    standardized_frac_rmse = None
    if getattr(result, 'matcher_diagnostics', None) is not None:
        standardized_frac_rmse = getattr(result.matcher_diagnostics, 'standardized_frac_rmse', None)
    return {
        'match': bool(result.match),
        'valid': bool(result.valid),
        'rmse': result.rmse,
        'frac_rmse': result.frac_rmse,
        'composition_match': result.composition_match,
        'requested_space_group_match': result.requested_space_group_match,
        'validity_reason': result.validity_reason,
        'standardized_frac_rmse': standardized_frac_rmse,
        'metric_source': 'sample_evaluation.evaluate_csp_reconstruction / StructureMatcher',
    }


def algo_times(case: GraphCase, t_value: float):
    dtype = case.gt_frac.dtype
    device = case.gt_frac.device
    t_graph = torch.as_tensor([[float(t_value)]], device=device, dtype=dtype)
    t_nodes = torch.full((int(case.gt_frac.shape[0]),), float(t_value), device=device, dtype=dtype)
    return t_graph, t_nodes


def make_algo_state_from_raw(*, f, v, l, atom_types, node_index, edge_node_index, t_graph, t_nodes) -> Algorithm19State:
    return Algorithm19State(
        f=f.detach().clone(),
        v=v.detach().clone(),
        l=l.detach().clone().reshape(-1),
        atom_types=atom_types.detach().clone(),
        node_index=node_index.detach().clone(),
        edge_node_index=edge_node_index.detach().clone(),
        t_graph=t_graph.detach().clone(),
        t_nodes=t_nodes.detach().clone(),
    )


def clean_prediction(state: Algorithm19State, *, variant: str = 'minus', coordinate_score_mode: str = 'direct') -> torch.Tensor:
    return kldm_clean_fractional_denoiser_Df(
        model=runner.model,
        f=state.f,
        v=state.v,
        l=state.l,
        atom_types=state.atom_types,
        t_graph=state.t_graph,
        t_nodes=state.t_nodes,
        node_index=state.node_index,
        edge_index=state.edge_node_index,
        variant=variant,
        coordinate_score_mode=coordinate_score_mode,
    )


def sample_gt_noisy_state(case: GraphCase, *, t_value: float, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 1000 * int(case.graph_id) + int(round(1000.0 * float(t_value))))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=case.gt_frac, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state


def sample_facit_kldm_em(case: GraphCase, *, n_steps: int = 1000, seed: int = SAMPLE_SEED):
    batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
    set_seed(int(seed) + 50000 * int(case.graph_id) + int(n_steps))
    f_t, v_t, l_t, a_t = runner.model.sample_CSP_algorithm3_facit(
        n_steps=int(n_steps),
        batch=batch_view,
        t_start=1.0,
        t_final=1.0e-6,
    )
    return {
        'f': f_t.detach().clone(),
        'v': v_t.detach().clone(),
        'l': l_t.detach().clone().reshape(-1),
        'a': a_t.detach().clone(),
        'batch_view': batch_view,
    }


def sample_template_q(case: GraphCase, payload, *, mode: str, seed: int = SAMPLE_SEED, dtype=None, device=None):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    dtype = case.gt_frac.dtype if dtype is None else dtype
    device = case.gt_frac.device if device is None else device
    if chart.total_dof == 0:
        return torch.empty((0,), device=device, dtype=dtype)
    mode_norm = str(mode).strip().lower()
    if mode_norm in {'crystalformer_oracle', 'oracle_structure', 'oracle'}:
        return torch.as_tensor(chart.q_ref, device=device, dtype=dtype).reshape(-1)
    if mode_norm in {'random_template', 'random'}:
        set_seed(int(seed) + 30000 * int(case.graph_id))
        return torch.rand((int(chart.total_dof),), device=device, dtype=dtype)
    raise ValueError(f'Unsupported q mode {mode!r}')


def build_template_f0_model(case: GraphCase, payload, *, q0: torch.Tensor):
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    q_eval = torch.remainder(q0.reshape(-1), 1.0) if q0.numel() else q0.reshape(-1)
    z_payload = chart.expand_q(q_eval, device=case.gt_frac.device, dtype=case.gt_frac.dtype)
    f0_model = map_payload_reference_chart_to_model_frame(z_payload, payload)
    return f0_model.detach().clone(), z_payload.detach().clone()


def sample_template_noisy_state(case: GraphCase, *, init_mode: str, t_value: float, seed: int = SAMPLE_SEED):
    payload = build_oracle_payload(case)
    q0 = sample_template_q(case, payload, mode=init_mode, seed=seed, dtype=case.gt_frac.dtype, device=case.gt_frac.device)
    f0_model, z0_payload = build_template_f0_model(case, payload, q0=q0)
    batch_view = make_single_graph_batch_view(case, ref_tensor=f0_model)
    set_seed(int(seed) + 70000 * int(case.graph_id) + int(round(1000.0 * float(t_value))) + (0 if init_mode == 'crystalformer_oracle' else 1))
    t_graph, t_nodes = algo_times(case, float(t_value))
    f_t, v_t, *_ = runner.model.tdm.sample_noisy_state(t=t_nodes, f0=f0_model, index=batch_view.batch)
    state = make_algo_state_from_raw(
        f=f_t,
        v=v_t,
        l=case.gt_l_feature,
        atom_types=case.atomic_numbers,
        node_index=batch_view.batch,
        edge_node_index=batch_view.edge_node_index,
        t_graph=t_graph,
        t_nodes=t_nodes,
    )
    return state, payload, q0, z0_payload


def q_distance(a: torch.Tensor, b: torch.Tensor) -> float:
    if a.numel() == 0 and b.numel() == 0:
        return 0.0
    diff = wrapdiff(a.reshape(-1), b.reshape(-1))
    return float(torch.sqrt(diff.square().mean()).detach().item())


def composition_to_formula_string(composition: dict[int, int]) -> str:
    counts = [int(v) for v in composition.values() if int(v) > 0]
    gcd_value = 0
    for value in counts:
        gcd_value = int(value) if gcd_value == 0 else math.gcd(int(gcd_value), int(value))
    gcd_value = gcd_value or 1
    parts = []
    for atomic_number in sorted(int(k) for k in composition):
        count = int(composition[atomic_number])
        if count <= 0:
            continue
        reduced = count // gcd_value
        symbol = Element.from_Z(int(atomic_number)).symbol
        parts.append(symbol if reduced == 1 else f'{symbol}{reduced}')
    return ''.join(parts)


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def config21(*, beta: float, alpha: float, q_opt_steps: int = Q_OPT_STEPS, post_renoise_acceptance: bool = True, debug_prints: bool = False, cf_grad_max_dims: int | None = None, cf_value_only_after_renoise: bool = False) -> Algorithm21Config:
    return Algorithm21Config(
        beta=float(beta),
        alpha=float(alpha),
        q_opt_steps=int(q_opt_steps),
        post_renoise_acceptance=bool(post_renoise_acceptance),
        debug_prints=bool(debug_prints),
        cf_grad_max_dims=cf_grad_max_dims,
        cf_value_only_after_renoise=bool(cf_value_only_after_renoise),
    )


def fit_clean_q(clean_f: torch.Tensor, *, case: GraphCase, payload, config: Algorithm21Config, cf_like=None, formula: str | None = None, q_init: torch.Tensor | None = None):
    z_payload = map_model_to_payload_reference_chart(clean_f, payload)
    return q_only_clean_cf_fit(
        z_payload=z_payload,
        payload=payload,
        t_nodes=torch.full((int(clean_f.shape[0]),), 0.3, device=clean_f.device, dtype=clean_f.dtype),
        lattice_feature=case.gt_l_feature,
        formula=formula,
        config=config,
        cf_likelihood=cf_like,
        q_init=q_init,
    )


loaded_graphs= [2] sg= [4]


## Notebook Scope

This notebook focuses on the new Algorithm24-specific questions. It reuses the stable KLDM-CPS/PyXtal/sample-evaluation machinery from Algorithm22 and the safe CrystalFormer runtime path from Algorithm23, but the tests are about EM-guided q-region extraction, local ranking, and one-step CPS survival.


In [5]:
audit_rows = [
    {'spec_version': 'old_algorithm22_markdown', 'area': 'A-F', 'status': 'audited', 'note': 'Used as baseline for earlier backend/notebook draft.'},
    {'spec_version': 'current_algorithm22_update', 'area': 'A-F', 'status': 'audited', 'note': 'This notebook tracks the current requested tests and implementation status.'},
]
source_rows = [
    {'source': 'src/PPR/ppr-main/ppr/projection.py', 'verified_point': 'CPS/PPR project-clean-then-continue principle.'},
    {'source': 'src/PPR/ppr-main/ppr/diffusion/samplers.py', 'verified_point': 'Projection lives inside the reverse-step flow after prediction.'},
    {'source': 'src/kldmPlus/symmetry/pcs_projection.py', 'verified_point': 'PyXtal-backed exact-composition template states already exist.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/sample.py', 'verified_point': 'CrystalFormer reduced-variable sampling remains expensive and is treated carefully.'},
    {'source': 'src/CrystalFormer/CrystalFormer-main/crystalformer/src/loss.py', 'verified_point': 'CrystalFormer coordinate NLL is used as a score, not the hard symmetry constraint.'},
]
symbol_rows = []
for name in [
    'algorithm22_projection_schedule', 'algorithm22_piecewise_alpha', 'algorithm22A_oracle_template_kldm_cps',
    'algorithm22B_ranked_kldm_cps', 'generate_candidates_22', 'sample_template_q_proposals_22',
    'expand_template_to_model_order_22', 'rank_projected_candidates_with_rule_22', 'safety_ok_22',
]:
    symbol_rows.append({'backend_symbol': name, 'available': bool(name in globals())})
display(pd.DataFrame(audit_rows))
display(pd.DataFrame(source_rows))
display(pd.DataFrame(symbol_rows))


,spec_version,area,status,note
0,old_algorithm22_markdown,A-F,audited,Used as baseline for earlier backend/notebook ...
1,current_algorithm22_update,A-F,audited,This notebook tracks the current requested tes...


,source,verified_point
0,src/PPR/ppr-main/ppr/projection.py,CPS/PPR project-clean-then-continue principle.
1,src/PPR/ppr-main/ppr/diffusion/samplers.py,Projection lives inside the reverse-step flow ...
2,src/kldmPlus/symmetry/pcs_projection.py,PyXtal-backed exact-composition template state...
3,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer reduced-variable sampling remain...
4,src/CrystalFormer/CrystalFormer-main/crystalfo...,CrystalFormer coordinate NLL is used as a scor...


,backend_symbol,available
0,algorithm22_projection_schedule,True
1,algorithm22_piecewise_alpha,True
2,algorithm22A_oracle_template_kldm_cps,True
3,algorithm22B_ranked_kldm_cps,True
4,generate_candidates_22,True
5,sample_template_q_proposals_22,True
6,expand_template_to_model_order_22,True
7,rank_projected_candidates_with_rule_22,True
8,safety_ok_22,True


## Runtime Preflight

CrystalFormer is still kept out of the notebook kernel during heavy work. The notebook uses subprocess smoke checks plus subprocess scoring helpers, then runs Algorithm23 logic around those safe energy evaluations.


In [6]:

rows = []
CF_READY = False
CF_COORDINATE_ONLY = np.nan
CF_RUNTIME_MODE = 'none'
CF_SMOKE_ERROR_TYPE = None
CF_SMOKE_ERROR_MESSAGE = None
CF_KERNEL_LIKE = None
CF_IMPORTS = {}
CF_WORKER_PROC = None
CF_WORKER_STARTUP_TIMEOUT = float(os.environ.get('KLDM_ALGO23_CF_STARTUP_TIMEOUT', '90'))
CF_LAST_ERROR_TYPE = None
CF_LAST_ERROR_MESSAGE = None
notebook_log('[algo23.preflight] cf checkpoint load start')


def _try_import_flag(module_name: str) -> bool:
    try:
        importlib.import_module(module_name)
        return True
    except Exception:
        return False


def _clear_cf_error() -> None:
    global CF_LAST_ERROR_TYPE, CF_LAST_ERROR_MESSAGE
    CF_LAST_ERROR_TYPE = None
    CF_LAST_ERROR_MESSAGE = None


def _record_cf_error(exc: Exception) -> None:
    global CF_LAST_ERROR_TYPE, CF_LAST_ERROR_MESSAGE, CF_SMOKE_ERROR_TYPE, CF_SMOKE_ERROR_MESSAGE
    CF_LAST_ERROR_TYPE = type(exc).__name__
    CF_LAST_ERROR_MESSAGE = ''.join(traceback.format_exception_only(type(exc), exc)).strip()
    CF_SMOKE_ERROR_TYPE = CF_LAST_ERROR_TYPE
    CF_SMOKE_ERROR_MESSAGE = CF_LAST_ERROR_MESSAGE


def _start_cf_worker_algo23():
    global CF_WORKER_PROC, CF_READY, CF_COORDINATE_ONLY, CF_RUNTIME_MODE, CF_SMOKE_ERROR_TYPE, CF_SMOKE_ERROR_MESSAGE
    if CF_WORKER_PROC is not None and getattr(CF_WORKER_PROC, 'poll', lambda: None)() is None:
        return CF_WORKER_PROC
    cmd = [
        sys.executable,
        str(ROOT / 'scripts' / 'algorithm23_cf_worker.py'),
        '--repo-root', str(ROOT),
        '--checkpoint', str(CF_CHECKPOINT),
    ]
    notebook_log('[algo23.worker] launching persistent CrystalFormer worker on first use')
    proc = subprocess.Popen(
        cmd,
        cwd=str(ROOT),
        stdin=subprocess.PIPE,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        bufsize=1,
        start_new_session=True,
    )
    msg = None
    raw_lines = []
    start_time = time.time()
    if proc.stdout is None:
        raise RuntimeError('CrystalFormer worker stdout unavailable')
    for _ in range(500):
        if (time.time() - start_time) > CF_WORKER_STARTUP_TIMEOUT:
            proc.kill()
            raise TimeoutError(f'CrystalFormer worker startup exceeded {CF_WORKER_STARTUP_TIMEOUT:.1f}s')
        line = proc.stdout.readline()
        if not line:
            break
        raw_lines.append(line.rstrip())
        stripped = line.strip()
        if not stripped:
            continue
        try:
            candidate = jsonlib.loads(stripped)
        except Exception:
            continue
        msg = candidate
        break
    if msg is None:
        stderr_tail = proc.stderr.read()[-1000:] if proc.stderr is not None else ''
        raise RuntimeError(f'CrystalFormer worker produced no JSON startup message. raw_lines={raw_lines[-5:]} stderr_tail={stderr_tail}')
    if not msg.get('ok'):
        raise RuntimeError(msg.get('error_message') or 'CrystalFormer worker failed to start')
    CF_WORKER_PROC = proc
    CF_READY = True
    CF_COORDINATE_ONLY = bool(msg.get('coordinate_only', True))
    CF_RUNTIME_MODE = 'subprocess_worker'
    _clear_cf_error()
    CF_SMOKE_ERROR_TYPE = None
    CF_SMOKE_ERROR_MESSAGE = None
    notebook_log('[algo23.worker] persistent CrystalFormer worker ready')
    return proc


def _cf_worker_request(request: dict[str, Any]) -> dict[str, Any]:
    proc = _start_cf_worker_algo23()
    if proc.stdin is None or proc.stdout is None:
        raise RuntimeError('CrystalFormer worker stdio unavailable')
    proc.stdin.write(jsonlib.dumps(request) + '\n')
    proc.stdin.flush()
    raw_lines = []
    msg = None
    for _ in range(500):
        line = proc.stdout.readline()
        if not line:
            break
        raw_lines.append(line.rstrip())
        stripped = line.strip()
        if not stripped:
            continue
        try:
            candidate = jsonlib.loads(stripped)
        except Exception:
            continue
        msg = candidate
        break
    if msg is None:
        stderr_tail = proc.stderr.read()[-1000:] if proc.stderr is not None else ''
        raise RuntimeError(f'CrystalFormer worker returned no JSON response. raw_lines={raw_lines[-5:]} stderr_tail={stderr_tail}')
    if not msg.get('ok'):
        raise RuntimeError(msg.get('error_message') or 'CrystalFormer worker request failed')
    return msg


try:
    ckpt_exists = Path(CF_CHECKPOINT).exists()
    for module_name in ('jax', 'haiku', 'optax'):
        CF_IMPORTS[module_name] = bool(_try_import_flag(module_name))

    runtime_preference = str(os.environ.get('KLDM_ALGO23_CF_RUNTIME_MODE', 'subprocess_primary')).strip().lower()
    if ckpt_exists and runtime_preference == 'kernel_primary':
        try:
            notebook_log('[algo23.preflight] trying in-kernel CrystalFormer runtime (opt-in)')
            CF_KERNEL_LIKE = CrystalFormerLikelihood(checkpoint_path=str(CF_CHECKPOINT))
            CF_READY = True
            CF_COORDINATE_ONLY = bool(CF_KERNEL_LIKE.coordinate_only)
            CF_RUNTIME_MODE = 'in_kernel_single_runtime'
            notebook_log('[algo23.preflight] in-kernel CrystalFormer runtime ready')
        except Exception as exc:
            CF_SMOKE_ERROR_TYPE = type(exc).__name__
            CF_SMOKE_ERROR_MESSAGE = ''.join(traceback.format_exception_only(type(exc), exc)).strip()
            notebook_log(f'[algo23.preflight] in-kernel runtime failed: {CF_SMOKE_ERROR_TYPE}: {CF_SMOKE_ERROR_MESSAGE}')
    elif ckpt_exists and runtime_preference == 'subprocess_primary':
        # Deferred worker startup keeps the notebook responsive; the real runtime
        # is launched on the first likelihood query.
        CF_READY = True
        CF_COORDINATE_ONLY = True
        CF_RUNTIME_MODE = 'deferred_subprocess_worker'
        notebook_log('[algo23.preflight] deferring CrystalFormer worker startup until first CF query')

    rows.append({
        'test': 'algorithm22_cf_checkpoint_load',
        'checkpoint_path': CF_CHECKPOINT,
        'checkpoint_exists': bool(ckpt_exists),
        'wrapper_ready': bool(CF_READY),
        'coordinate_only': CF_COORDINATE_ONLY,
        'runtime_mode': CF_RUNTIME_MODE,
        'python_executable': sys.executable,
        'jax_importable': bool(CF_IMPORTS.get('jax')),
        'haiku_importable': bool(CF_IMPORTS.get('haiku')),
        'optax_importable': bool(CF_IMPORTS.get('optax')),
        'smoke_error_type': CF_SMOKE_ERROR_TYPE,
        'smoke_error_message': CF_SMOKE_ERROR_MESSAGE,
        'PASS': bool(CF_READY),
        'status': 'INFO' if CF_READY else ('MISSING_CHECKPOINT' if not ckpt_exists else 'CF_RUNTIME_FAILED'),
    })
except Exception as exc:
    rows.append(error_row('algorithm22_cf_checkpoint_load', 'na', exc, 'cf_checkpoint_load', checkpoint_path=CF_CHECKPOINT))
safe_display_sorted(dataframe_result('algorithm22_cf_checkpoint_load', rows), ['checkpoint_path'])


[algo23.preflight] cf checkpoint load start


[algo23.preflight] deferring CrystalFormer worker startup until first CF query


,test,checkpoint_path,checkpoint_exists,wrapper_ready,coordinate_only,runtime_mode,python_executable,jax_importable,haiku_importable,optax_importable,smoke_error_type,smoke_error_message,PASS,status
0,algorithm22_cf_checkpoint_load,/workspace/notebooks/epoch_046000.pkl,True,True,True,deferred_subprocess_worker,/workspace/.venv/bin/python,True,True,True,None,None,True,INFO


In [7]:

ALGO22_CFG = Algorithm22Config(
    schedule=algo22_backend.Algorithm22ScheduleConfig(
        n_pc_steps=800,
        projection_interval=50,
        p_start=0.625,
        alpha_max=0.25,
    ),
    top_branches=1,
    state_return_mode='preserve_velocity_shift',
    lattice_projection=False,
    post_acceptance=True,
)
ALGO22_SCHEDULE = algorithm22_projection_schedule(schedule=ALGO22_CFG.schedule)
ALGO22_PROJECTION_STEPS = [int(pt.step) for pt in ALGO22_SCHEDULE if pt.project]
ALGO22_LOCAL_STEPS = [600, 750]
RUN_ALGO22_CF_HEAVY = str(os.environ.get('RUN_ALGO22_CF_HEAVY', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
ALGO22_PHASEA_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PHASEE_SOURCES = ['baseline', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])


def algo22_step_to_progress(step: int, n_pc_steps: int = 800) -> float:
    return float(step) / float(max(1, n_pc_steps - 1))


def algo22_step_to_t(step: int, n_pc_steps: int = 800) -> float:
    return float(max(1.0e-6, 1.0 - algo22_step_to_progress(step, n_pc_steps=n_pc_steps)))


def oracle_signature(case: GraphCase):
    payload = build_oracle_payload(case)
    return tuple(sorted((int(z), str(label)) for z, label in zip(payload.anchor_atomic_numbers.tolist(), payload.wyckoff_letters)))


def template_signature(template) -> tuple:
    return flatten_site_signature(template)


def candidate_rmse_to_gt(case: GraphCase, frac_coords_model_order: torch.Tensor) -> float:
    return float(evaluate_arrays(case, frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)['frac_rmse'])


def safe_metric_float(value, default: float = float('nan')) -> float:
    if value is None:
        return float(default)
    try:
        out = float(value)
    except Exception:
        return float(default)
    return out if np.isfinite(out) else float(default) if not np.isnan(default) else out


def safe_metric_bool(value) -> bool:
    return bool(False if value is None else value)


def eval_metric_bundle(ev: dict[str, Any]) -> dict[str, Any]:
    return {
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
        'composition_match': safe_metric_bool(ev.get('composition_match')),
        'requested_space_group_match': safe_metric_bool(ev.get('requested_space_group_match')),
        'standardized_frac_rmse': safe_metric_float(ev.get('standardized_frac_rmse')),
    }


def topk_min(values, k: int):
    vals = [float(v) for v in values if np.isfinite(float(v))]
    if not vals:
        return float('nan')
    return float(min(sorted(vals)[: max(1, int(k))]))


ALGO22_CF_SCORE_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_CF_SCORE_CANDIDATE_LIMIT', '4'))
ALGO22_CF_AUG_BASE_LIMIT = int(os.environ.get('ALGO22_CF_AUG_BASE_LIMIT', '1'))


def cf_nll_eval_batch(*, items, label: str) -> list[float]:
    if not CF_READY or not items:
        return [float('nan')] * len(items)
    tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
    tmp_dir.mkdir(parents=True, exist_ok=True)
    safe_label = ''.join(ch if ch.isalnum() or ch in {'-', '_'} else '_' for ch in str(label))[-120:]
    in_path = tmp_dir / f'{safe_label}_batch.pkl'
    out_path = tmp_dir / f'{safe_label}_batch.json'
    packed = []
    for item in items:
        packed.append({
            'symmetry_payload': item['payload'],
            'q': np.asarray(torch.as_tensor(item['q']).detach().cpu(), dtype=float),
            'lattice_feature': np.asarray(torch.as_tensor(item['lattice_feature']).detach().cpu(), dtype=float),
            'formula': item.get('formula'),
        })
    with in_path.open('wb') as handle:
        pickle.dump({
            'repo_root': str(ROOT),
            'checkpoint_path': str(CF_CHECKPOINT),
            'items': packed,
        }, handle)
    cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_score_batch_once.py'), '--input', str(in_path), '--output', str(out_path)]
    notebook_log(f'[cf-eval-batch] subprocess start {label} n={len(items)}')
    completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
    result = {}
    if out_path.exists():
        with out_path.open('r', encoding='utf-8') as handle:
            result = jsonlib.load(handle)
    if completed.returncode != 0 or not result.get('ok'):
        notebook_log(f"[cf-eval-batch] ERROR {label} returncode={completed.returncode} stderr_tail={completed.stderr[-500:]}")
        raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer batch subprocess failed')
    values = [float(v) for v in result.get('values', [])]
    if len(values) != len(items):
        raise RuntimeError(f'CrystalFormer batch returned {len(values)} values for {len(items)} items')
    return values


def cf_formula_for_case(case: GraphCase) -> str | None:
    return CF_FORMULA or composition_to_formula_string(case.composition)


def q_strategy_label(strategy: str) -> str:
    mapping = {
        'pcs_anchor': 'pcs_anchor',
        'uniform': 'uniform',
        'sobol': 'sobol',
        'special-position-biased': 'special',
        'local': 'local',
        'small_local_perturbations': 'local',
    }
    return mapping.get(str(strategy), str(strategy))


def candidate_cf_stats(candidates, *, cf_mode: str):
    cf_vals = [float(c.cf_nll) for c in candidates if np.isfinite(float(c.cf_nll))]
    sampled = sum(1 for c in candidates if 'cf_sample_index' in (c.metadata or {}))
    return {
        'cf_active': bool(CF_READY and cf_mode != 'none'),
        'cf_mode': str(cf_mode),
        'cf_nll_finite': bool(len(cf_vals) > 0),
        'num_cf_q_samples': int(sampled),
        'num_cf_scored': int(len(cf_vals)),
        'cf_nll_min': float(np.min(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_max': float(np.max(cf_vals)) if cf_vals else float('nan'),
        'cf_nll_has_nan': bool(any(not np.isfinite(float(c.cf_nll)) for c in candidates)),
    }


def make_algo22_cfg(*, p_start: float | None = None, q_sampling_strategies=None, num_q_per_template: int | None = None, top_branches: int | None = None, cf_sample_k: int | None = None, source_mode: str | None = None, state_return_mode: str | None = None):
    cfg = ALGO22_CFG
    if p_start is not None:
        cfg = replace(cfg, schedule=replace(cfg.schedule, p_start=float(p_start)))
    if q_sampling_strategies is not None:
        cfg = replace(cfg, pyxtal_q_sampling_strategies=tuple(q_sampling_strategies))
    if num_q_per_template is not None:
        cfg = replace(cfg, pyxtal_q_samples_per_template=int(num_q_per_template))
    if top_branches is not None:
        cfg = replace(cfg, top_branches=int(top_branches))
    if cf_sample_k is not None:
        cfg = replace(cfg, cf_sample_k=int(cf_sample_k))
    if source_mode is not None:
        cfg = replace(cfg, crystalformer_template_mode=str(source_mode))
    if state_return_mode is not None:
        cfg = replace(cfg, state_return_mode=str(state_return_mode))
    return cfg


def sample_q_candidates_eval(*, payload, lattice_feature, formula, K: int, seed: int, top_p: float = 1.0, temperature: float = 1.0):
    if not CF_READY:
        return [], []
    use_subprocess = str(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS', 'true')).strip().lower() in {'1', 'true', 'yes', 'on'}
    if use_subprocess:
        tmp_dir = ROOT / 'notebooks' / '.algo21_cf_eval_tmp'
        tmp_dir.mkdir(parents=True, exist_ok=True)
        in_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.pkl'
        out_path = tmp_dir / f'algorithm22_sample_q_K{int(K)}_seed{int(seed)}.json'
        with in_path.open('wb') as handle:
            pickle.dump({
                'repo_root': str(ROOT),
                'checkpoint_path': str(CF_CHECKPOINT),
                'symmetry_payload': payload,
                'lattice_feature': np.asarray(torch.as_tensor(lattice_feature).detach().cpu(), dtype=float),
                'formula': formula,
                'K': int(K),
                'top_p': float(top_p),
                'temperature': float(temperature),
                'seed': int(seed),
            }, handle)
        cmd = [sys.executable, str(ROOT / 'scripts' / 'algorithm22_cf_sample_q_once.py'), '--input', str(in_path), '--output', str(out_path)]
        completed = subprocess.run(cmd, cwd=str(ROOT), text=True, capture_output=True, timeout=int(os.environ.get('KLDM_ALGO21_CF_SUBPROCESS_TIMEOUT', '900')), start_new_session=True)
        result = {}
        if out_path.exists():
            with out_path.open('r', encoding='utf-8') as handle:
                result = jsonlib.load(handle)
        if completed.returncode != 0 or not result.get('ok'):
            raise RuntimeError(result.get('error_message') or completed.stderr[-500:] or 'CrystalFormer q sampling subprocess failed')
        q_candidates = [torch.as_tensor(q_i, dtype=torch.as_tensor(lattice_feature).dtype, device=torch.as_tensor(lattice_feature).device) for q_i in result['q_candidates']]
        cf_nll = [float(v) for v in result.get('cf_nll', [])]
        return q_candidates, cf_nll
    return sample_q_from_crystalformer(
        payload=payload,
        lattice_feature=torch.as_tensor(lattice_feature),
        formula=formula,
        cf_likelihood=None,
        K=int(K),
        top_p=float(top_p),
        temperature=float(temperature),
        seed=int(seed),
    )


def generate_source_candidates(case: GraphCase, state, f0_hat: torch.Tensor, *, source: str, cfg):
    formula = cf_formula_for_case(case)
    cf_mode = 'none'
    if source in {'pyxtal_cf_score', 'pyxtal_cf_q_augmented'} and not RUN_ALGO22_CF_HEAVY:
        return tuple(), {
            'cf_active': bool(CF_READY),
            'cf_mode': 'disabled',
            'cf_nll_finite': False,
            'num_cf_q_samples': 0,
            'num_cf_scored': 0,
            'cf_nll_min': float('nan'),
            'cf_nll_max': float('nan'),
            'cf_nll_has_nan': False,
        }
    if source == 'pyxtal_only':
        candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src={source}',
        )
        candidates = tuple(candidates[: max(1, int(ALGO22_PYXTAL_CANDIDATE_LIMIT))])
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_score':
        cf_mode = 'score_only'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_score',
        )
        base_candidates = tuple(base_candidates[: max(1, int(ALGO22_CF_SCORE_CANDIDATE_LIMIT))])
        nll_values = cf_nll_eval_batch(items=[
            {'payload': cand.payload, 'q': cand.q_init, 'lattice_feature': cand.lattice_feature, 'formula': formula}
            for cand in base_candidates
        ], label=f'g{case.graph_id}_cfscore_batch') if base_candidates else []
        scored = []
        for cand, nll in zip(base_candidates, nll_values):
            scored.append(replace(cand, source='pyxtal_cf_score', cf_nll=float(nll), metadata={**(cand.metadata or {}), 'cf_scored': True}))
        candidates = tuple(scored)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    if source == 'pyxtal_cf_q_augmented':
        cf_mode = 'q_augmented'
        base_candidates = generate_candidates_22(
            f0_hat=f0_hat,
            state=state,
            space_group=case.requested_sg,
            source='pyxtal_only',
            lattice_transform=runner.lattice_transform,
            config=cfg,
            formula=formula,
            cf_likelihood=None,
            debug_label=f'g={case.graph_id} src=pyxtal_only_for_cf_aug',
        )
        augmented = []
        for base_idx, cand in enumerate(base_candidates[: min(len(base_candidates), int(ALGO22_CF_AUG_BASE_LIMIT))]):
            augmented.append(replace(cand, source='pyxtal_cf_q_augmented', metadata={**(cand.metadata or {}), 'cf_aug_base': True}))
            q_candidates, cf_nll_values = sample_q_candidates_eval(
                payload=cand.payload,
                lattice_feature=cand.lattice_feature,
                formula=formula,
                K=int(cfg.cf_sample_k),
                top_p=float(cfg.cf_top_p),
                temperature=float(cfg.cf_temperature),
                seed=int(cfg.cf_sampler_seed) + int(case.graph_id) * 1000 + int(base_idx),
            )
            for sample_idx, (q_i, nll) in enumerate(zip(q_candidates, cf_nll_values)):
                try:
                    aug = materialize_candidate_from_template_22(
                        template=cand.template,
                        q=q_i.reshape(-1),
                        state=state,
                        lattice_matrix=cand.lattice_matrix,
                        lattice_feature=cand.lattice_feature,
                        source='pyxtal_cf_q_augmented',
                        metadata={**(cand.metadata or {}), 'cf_sample_index': int(sample_idx), 'cf_aug_base_index': int(base_idx)},
                        cf_nll=float(nll),
                        spacegroup=case.requested_sg,
                    )
                    augmented.append(aug)
                except Exception:
                    continue
        candidates = tuple(augmented)
        return candidates, candidate_cf_stats(candidates, cf_mode=cf_mode)
    raise ValueError(f'Unsupported source: {source}')


def fit_candidate_to_clean(case: GraphCase, state, f0_hat: torch.Tensor, candidate, cfg):
    fitted = algo22_backend.fit_q_to_template(
        candidate=candidate,
        target_f0=f0_hat,
        target_atomic_numbers=state.atom_types,
        q_init=candidate.q_init,
        lambda_init=0.0,
        q_opt_steps=int(cfg.q_opt_steps),
        q_lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    return fitted


ALGO22_LOCAL_SOURCES = ['no_correction', 'oracle_template', 'pyxtal_only'] + (['pyxtal_cf_score', 'pyxtal_cf_q_augmented'] if RUN_ALGO22_CF_HEAVY and CF_READY else [])
ALGO22_PYXTAL_CANDIDATE_LIMIT = int(os.environ.get('ALGO22_PYXTAL_CANDIDATE_LIMIT', '8'))
REPLAY_CACHE = {}


def fit_oracle_payload_from_q_init(case: GraphCase, state, f0_hat: torch.Tensor, payload, *, q_init, cfg, init_label: str):
    q_gt = torch.as_tensor(algo22_backend._get_wyckoff_dof_chart(payload).q_ref, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    z_hat = model_to_payload(f_model=f0_hat, payload=payload)
    q_init_tensor = None if q_init is None else torch.as_tensor(q_init, device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    fit = fit_q_to_clean_prediction(
        z_hat=z_hat,
        payload=payload,
        t_nodes=state.t_nodes,
        lattice_feature=state.l,
        q_init=q_init_tensor,
        q_init_mode='random' if q_init_tensor is None else 'oracle_structure',
        steps=int(cfg.q_opt_steps),
        lr=float(cfg.q_lr),
        grad_clip=float(cfg.grad_clip),
    )
    f_proj = payload_to_model(z_payload=fit.z_proj_payload, payload=payload)
    ev = evaluate_arrays(case, f_proj, case.gt_l_feature, case.atomic_numbers)
    q_init_dist = float('nan') if q_init_tensor is None or q_init_tensor.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(q_init_tensor.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    q_fit_dist = float('nan') if fit.q_star.numel() != q_gt.numel() else float(torch.sqrt(cf_backend.torus_mse(fit.q_star.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item())
    return {
        'init_label': str(init_label),
        'q_init_distance_to_GT': q_init_dist,
        'q_fit_distance_to_GT': q_fit_dist,
        'witness_after': float(fit.witness_sin),
        'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
        'rmse': safe_metric_float(ev.get('rmse')),
        'match': safe_metric_bool(ev.get('match')),
        'valid': safe_metric_bool(ev.get('valid')),
    }


def ranked_summary(case: GraphCase, ranked) -> dict[str, Any]:
    if not ranked:
        return {
            'top1_frac_rmse': float('nan'),
            'top1_rmse': float('nan'),
            'top1_match': False,
            'top1_valid': False,
            'top3_min_frac_rmse': float('nan'),
        }
    top1_eval = evaluate_arrays(case, ranked[0].frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
    top3_min = min([candidate_rmse_to_gt(case, item.frac_coords_model_order) for item in ranked[:3]], default=float('nan'))
    return {
        'top1_frac_rmse': float(top1_eval['frac_rmse']),
        'top1_rmse': float(top1_eval['rmse']),
        'top1_match': bool(top1_eval['match']),
        'top1_valid': bool(top1_eval['valid']),
        'top3_min_frac_rmse': float(top3_min),
    }


def replay_source(case: GraphCase, *, source: str, cfg, alpha_override=None):
    cache_key = (int(case.graph_id), str(source), repr(cfg), None if alpha_override is None else float(alpha_override))
    if cache_key in REPLAY_CACHE:
        return REPLAY_CACHE[cache_key]
    rows = []
    baseline_rows = []
    payload = build_oracle_payload(case)
    for step in ALGO22_PROJECTION_STEPS:
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=cfg)
        base_eval = evaluate_arrays(case, f0_hat, case.gt_l_feature, case.atomic_numbers)
        base_witness = float(oracle_template_witness_22(state=state, model=runner.model, payload=payload, config=cfg))
        baseline_rows.append({'step': step, 'frac_rmse': safe_metric_float(base_eval.get('frac_rmse')), 'rmse': safe_metric_float(base_eval.get('rmse')), 'match': safe_metric_bool(base_eval.get('match')), 'valid': safe_metric_bool(base_eval.get('valid')), 'witness': base_witness})
        alpha = float(algorithm22_piecewise_alpha(algo22_step_to_progress(step)) if alpha_override is None else alpha_override)
        if source == 'baseline':
            rows.append({**baseline_rows[-1], 'accepted': True, 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        if source == 'oracle_template':
            branch = oracle_template_projection_22(state=state, model=runner.model, payload=payload, alpha=alpha, beta=1.0, config=cfg)
            ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
            rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), 'cf_active': False, 'cf_mode': 'none', 'cf_nll_finite': False, 'num_cf_q_samples': 0})
            continue
        candidates, cf_stats = generate_source_candidates(case, state, f0_hat, source=source, cfg=cfg)
        ranked = rank_candidates_against_clean_estimate_22(f0_hat=f0_hat, state=state, candidates=candidates, config=cfg)
        if not ranked:
            rows.append({'step': step, 'frac_rmse': float('nan'), 'rmse': float('nan'), 'match': False, 'valid': False, 'witness': float('inf'), 'accepted': False, **cf_stats})
            continue
        branch = algo22_backend.branch_survival_step(state=state, model=runner.model, projected=ranked[0], alpha=alpha, beta=1.0, candidate_pool=tuple(candidates), config=cfg)
        ev = evaluate_arrays(case, branch.f0_hat_after, case.gt_l_feature, case.atomic_numbers)
        rows.append({'step': step, 'frac_rmse': safe_metric_float(ev.get('frac_rmse')), 'rmse': safe_metric_float(ev.get('rmse')), 'match': safe_metric_bool(ev.get('match')), 'valid': safe_metric_bool(ev.get('valid')), 'witness': float(branch.witness_after), 'accepted': bool(branch.accepted), **cf_stats})
    baseline_frac = float(np.nanmean([r['frac_rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_rmse = float(np.nanmean([r['rmse'] for r in baseline_rows])) if baseline_rows else float('nan')
    baseline_witness = float(np.nanmean([r['witness'] for r in baseline_rows])) if baseline_rows else float('nan')
    finite_rows = [r for r in rows if np.isfinite(float(r['frac_rmse']))]
    best_row = min(finite_rows, key=lambda r: float(r['frac_rmse'])) if finite_rows else None
    out = {
        'rows': rows,
        'baseline_rows': baseline_rows,
        'frac_rmse': float(np.nanmean([r['frac_rmse'] for r in rows])) if rows else float('nan'),
        'rmse': float(np.nanmean([r['rmse'] for r in rows])) if rows else float('nan'),
        'match_rate': float(np.mean([bool(r['match']) for r in rows])) if rows else float('nan'),
        'valid_rate': float(np.mean([bool(r['valid']) for r in rows])) if rows else float('nan'),
        'group_witness': float(np.nanmean([r['witness'] for r in rows])) if rows else float('nan'),
        'accepted_fraction': float(np.mean([bool(r['accepted']) for r in rows])) if rows else float('nan'),
        'projection_count': int(len(rows)),
        'best_step': int(best_row['step']) if best_row is not None else -1,
        'best_step_frac_rmse': float(best_row['frac_rmse']) if best_row is not None else float('nan'),
        'best_step_rmse_to_GT': float(best_row['rmse']) if best_row is not None else float('nan'),
        'best_step_match': bool(best_row['match']) if best_row is not None else False,
        'best_step_valid': bool(best_row['valid']) if best_row is not None else False,
        'beats_baseline': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_witness': bool(np.nanmean([r['witness'] for r in rows]) < baseline_witness) if rows and np.isfinite(baseline_witness) else False,
        'improves_frac_rmse': bool(np.nanmean([r['frac_rmse'] for r in rows]) < baseline_frac) if rows and np.isfinite(baseline_frac) else False,
        'improves_rmse': bool(np.nanmean([r['rmse'] for r in rows]) < baseline_rmse) if rows and np.isfinite(baseline_rmse) else False,
    }
    REPLAY_CACHE[cache_key] = out
    return out


In [8]:
Algorithm24Config = algo24_backend.Algorithm24Config
Algorithm24EMConfig = algo24_backend.Algorithm24EMConfig
Algorithm24RegionConfig = algo24_backend.Algorithm24RegionConfig
Algorithm24SelectionConfig = algo24_backend.Algorithm24SelectionConfig
algorithm24_run_facit_em_lookahead = algo24_backend.algorithm24_run_facit_em_lookahead
algorithm24_generate_template_candidates = algo24_backend.algorithm24_generate_template_candidates
algorithm24_explain_templates = algo24_backend.algorithm24_explain_templates
algorithm24_build_q_region = algo24_backend.algorithm24_build_q_region
algorithm24_choose_pre_projection_candidate = algo24_backend.algorithm24_choose_pre_projection_candidate
algorithm24_select_by_post_update_survival = algo24_backend.algorithm24_select_by_post_update_survival
select_by_anchor_then_cf = algo24_backend.select_by_anchor_then_cf

ALGO24_USE_CF = False
ALGO24_CFG = Algorithm24Config(
    algo22=ALGO22_CFG,
    em=Algorithm24EMConfig(
        lookahead_steps=300,
        t_final=1.0e-6,
    ),
    region=Algorithm24RegionConfig(
        lambdas=(-0.25, 0.0, 0.25, 0.50, 0.75, 1.0, 1.25),
        perturb_sigmas=(0.0, 0.0025, 0.005),
        perturbations_per_lambda=1,
        relative_slack_now=float(os.environ.get('ALGO24_RHO_NOW', '0.10')),
        absolute_slack_now=1.0e-6,
        relative_slack_em=float(os.environ.get('ALGO24_RHO_EM', '0.10')),
        absolute_slack_em=1.0e-6,
        em_residual_gate=float(os.environ.get('ALGO24_EM_RESIDUAL_GATE', '0.02')),
        cross_slack_abs=float(os.environ.get('ALGO24_CROSS_SLACK', '0.002')),
        anchor_eps=float(os.environ.get('ALGO24_ANCHOR_EPS', '1e-6')),
        anchor_tiebreak_delta=float(os.environ.get('ALGO24_ANCHOR_DELTA', '0.05')),
        seed=int(SAMPLE_SEED),
    ),
    selection=Algorithm24SelectionConfig(
        lambda_em=1.0,
        lambda_shift=float(os.environ.get('ALGO24_LAMBDA_SHIFT', '0.01')),
        shift_cap=0.02,
        prefer_cf_within_anchor_band=False,
    ),
)
ALGO24_LATE_STEPS = [500]
ALGO24_ALPHA_VALUES = [0.03]
ALGO24_EM_STEPS = [300]
ALGO24_EM_STABILITY_SEEDS = [SAMPLE_SEED, SAMPLE_SEED + 1]
ALGO24_TOP_K_TEMPLATES = [1]
ALGO24_TEMPLATE_LIMIT = int(os.environ.get('ALGO24_TEMPLATE_LIMIT', '6'))
ALGO24_SAFETY_U_THRESH = float(os.environ.get('ALGO24_SAFETY_U_THRESH', '0.01'))
ALGO24_SAFETY_THRESHOLDS = [1e-4, 1e-2]
ALGO24_STRICT_TOL = float(os.environ.get('ALGO24_STRICT_TOL', '1e-6'))
ALGO24_GATE_RHOS = [0.01, 0.10]
ALGO24_SAMPLER_SEEDS = [int(SAMPLE_SEED)]
ALGO24_CACHE: dict[tuple[Any, ...], Any] = {}


def make_algo24_cfg(*, em_steps: int | None = None, em_gate: float | None = None, cross_slack: float | None = None, anchor_eps: float | None = None, lambda_em: float | None = None, rho_now: float | None = None, rho_em: float | None = None, anchor_delta: float | None = None, lambda_shift: float | None = None):
    return Algorithm24Config(
        algo22=ALGO22_CFG,
        em=Algorithm24EMConfig(
            lookahead_steps=int(ALGO24_CFG.em.lookahead_steps if em_steps is None else em_steps),
            t_final=float(ALGO24_CFG.em.t_final),
        ),
        region=Algorithm24RegionConfig(
            lambdas=tuple(ALGO24_CFG.region.lambdas),
            perturb_sigmas=tuple(ALGO24_CFG.region.perturb_sigmas),
            perturbations_per_lambda=int(ALGO24_CFG.region.perturbations_per_lambda),
            relative_slack_now=float(ALGO24_CFG.region.relative_slack_now if rho_now is None else rho_now),
            absolute_slack_now=float(ALGO24_CFG.region.absolute_slack_now),
            relative_slack_em=float(ALGO24_CFG.region.relative_slack_em if rho_em is None else rho_em),
            absolute_slack_em=float(ALGO24_CFG.region.absolute_slack_em),
            em_residual_gate=float(ALGO24_CFG.region.em_residual_gate if em_gate is None else em_gate),
            cross_slack_abs=float(ALGO24_CFG.region.cross_slack_abs if cross_slack is None else cross_slack),
            anchor_eps=float(ALGO24_CFG.region.anchor_eps if anchor_eps is None else anchor_eps),
            anchor_tiebreak_delta=float(ALGO24_CFG.region.anchor_tiebreak_delta if anchor_delta is None else anchor_delta),
            seed=int(SAMPLE_SEED),
        ),
        selection=Algorithm24SelectionConfig(
            lambda_em=float(ALGO24_CFG.selection.lambda_em if lambda_em is None else lambda_em),
            lambda_shift=float(ALGO24_CFG.selection.lambda_shift if lambda_shift is None else lambda_shift),
            shift_cap=float(ALGO24_CFG.selection.shift_cap),
            prefer_cf_within_anchor_band=False,
        ),
    )


def recover_oracle_template_from_payload(case: GraphCase, payload):
    templates = extract_wyckoff_templates(
        space_group_number=int(payload.spacegroup),
        atomic_numbers=case.atomic_numbers,
        max_templates=64,
        quick=False,
    )
    signature = payload.site_signature
    template = None
    for candidate in templates:
        if flatten_site_signature(candidate) == signature:
            template = candidate
            break
    if template is None:
        raise RuntimeError(
            f"No oracle template matched payload signature for graph={case.graph_id} sg={int(payload.spacegroup)} signature={signature!r}."
        )
    q_ref = recover_template_free_vars_from_anchor_entries(template, payload.anchor_entries)
    return template, q_ref.to(device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)


def oracle_template_candidate(case: GraphCase, state, payload, f0_hat: torch.Tensor):
    template, q_ref = recover_oracle_template_from_payload(case, payload)
    q_ref = q_ref.to(device=f0_hat.device, dtype=f0_hat.dtype).reshape(-1)
    lattice_matrix = decode_state_cell_matrix_22(state=state, lattice_transform=runner.lattice_transform)
    return materialize_candidate_from_template_22(
        template=template,
        q=q_ref,
        state=state,
        lattice_matrix=lattice_matrix,
        lattice_feature=state.l,
        source='oracle_template',
        metadata={'mode': 'oracle_template'},
        spacegroup=int(payload.spacegroup),
    )


def algorithm24_oracle_geom_fit(case: GraphCase, state, payload, target_f0: torch.Tensor, cfg24=ALGO24_CFG):
    cand = oracle_template_candidate(case, state, payload, target_f0)
    return fit_candidate_to_clean(case, state, target_f0, cand, cfg24.algo22)


def strict_improves(new_value, old_value, tol: float = ALGO24_STRICT_TOL) -> bool:
    if new_value is None or old_value is None:
        return False
    if not np.isfinite(float(new_value)) or not np.isfinite(float(old_value)):
        return False
    return float(new_value) < float(old_value) - float(tol)


def compare_metric(new_value, old_value, tol: float = ALGO24_STRICT_TOL) -> str:
    if strict_improves(new_value, old_value, tol=tol):
        return 'better'
    if strict_improves(old_value, new_value, tol=tol):
        return 'worse'
    return 'tie'


def evaluate_frac(case: GraphCase, frac: torch.Tensor) -> dict[str, Any]:
    return evaluate_arrays(case, frac, case.gt_l_feature, case.atomic_numbers)


def finite_min(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(min(vals)) if vals else float('nan')


def finite_mean(values) -> float:
    vals = [float(v) for v in values if np.isfinite(float(v))]
    return float(np.mean(vals)) if vals else float('nan')


def _cache_key(*parts: Any) -> tuple[Any, ...]:
    return tuple(parts)


def _cached(key: tuple[Any, ...], fn):
    if key not in ALGO24_CACHE:
        ALGO24_CACHE[key] = fn()
    return ALGO24_CACHE[key]


def cf_formula_for_case(case: GraphCase) -> str:
    return composition_to_formula_string(case.composition)


def run_algo24_em_lookahead(case: GraphCase, step: int, *, em_steps: int, seed: int | None = None, cfg24=ALGO24_CFG):
    def _runner():
        state, f0_hat = get_state_and_clean(case, step)
        batch = make_single_graph_batch_view(case)
        lookahead = algorithm24_run_facit_em_lookahead(
            model=runner.model,
            batch=batch,
            state=state,
            steps=int(em_steps),
            t_final=float(cfg24.em.t_final),
        )
        return {'state': state, 'f0_hat': f0_hat, 'lookahead': lookahead}
    return _cached(_cache_key('em_lookahead', case.graph_id, step, em_steps, seed, cfg24.em.t_final), _runner)


def get_state_and_clean(case: GraphCase, step: int):
    def _runner():
        state = sample_gt_noisy_state(case, t_value=algo22_step_to_t(step))
        f0_hat = clean_fractional_estimate_22(state=state, model=runner.model, config=ALGO22_CFG)
        return state, f0_hat
    return _cached(_cache_key('state_clean', case.graph_id, step), _runner)


def get_pyxtal_ranked(case: GraphCase, step: int, *, top_k: int = 1, cfg24=ALGO24_CFG):
    def _runner():
        state, f0_hat = get_state_and_clean(case, step)
        ranked = algo24_backend.algorithm24_generate_template_candidates(
            f0_hat=f0_hat,
            state=state,
            space_group=int(case.requested_sg),
            lattice_transform=runner.lattice_transform,
            formula=cf_formula_for_case(case),
            cf_likelihood=None,
            config=cfg24,
            source='pyxtal_only',
            debug_label=f'g{case.graph_id}_algo24_s{step}',
        )
        return state, f0_hat, tuple(ranked[:top_k])
    return _cached(_cache_key('pyxtal_ranked', case.graph_id, step, top_k, cfg24.region.relative_slack_now, cfg24.region.relative_slack_em), _runner)


def explain_case_step(case: GraphCase, step: int, em_steps: int, *, top_k: int = 1, cfg24=ALGO24_CFG):
    def _runner():
        state, f0_hat, ranked = get_pyxtal_ranked(case, step, top_k=top_k, cfg24=cfg24)
        lookahead = run_algo24_em_lookahead(case, step, em_steps=em_steps, cfg24=cfg24)['lookahead']
        explanations = algorithm24_explain_templates(
            candidates=tuple(ranked),
            target_now=f0_hat,
            target_em=lookahead.f0_em,
            target_atomic_numbers=state.atom_types,
            config=cfg24,
        )
        return {'state': state, 'f0_hat': f0_hat, 'ranked': ranked, 'lookahead': lookahead, 'explanations': explanations}
    return _cached(_cache_key('explain', case.graph_id, step, em_steps, top_k, cfg24.region.relative_slack_now, cfg24.region.relative_slack_em), _runner)


def build_region(case: GraphCase, step: int, em_steps: int, explanation, *, use_cf: bool, cfg24=ALGO24_CFG):
    def _runner():
        state, f0_hat = get_state_and_clean(case, step)
        lookahead = run_algo24_em_lookahead(case, step, em_steps=em_steps, cfg24=cfg24)['lookahead']
        region = algorithm24_build_q_region(
            explanation=explanation,
            target_now=f0_hat,
            target_em=lookahead.f0_em,
            target_atomic_numbers=state.atom_types,
            lattice_feature=state.l,
            cf_energy_fn=None,
            cf_formula=cf_formula_for_case(case),
            config=cfg24,
        )
        if not use_cf or not ALGO24_USE_CF or not CF_READY:
            return region
        accepted = [cand for cand in region if cand.accepted_by_region]
        values = cf_nll_eval_batch(
            items=[{'payload': cand.explanation.candidate.payload, 'q': cand.q, 'lattice_feature': state.l, 'formula': cf_formula_for_case(case)} for cand in accepted],
            label=f'g{case.graph_id}_algo24_region_s{step}_em{em_steps}',
        ) if accepted else []
        out = []
        idx = 0
        for cand in region:
            if cand.accepted_by_region:
                out.append(replace(cand, cf_nll=safe_metric_float(values[idx])))
                idx += 1
            else:
                out.append(cand)
        return tuple(out)
    return _cached(_cache_key('region', case.graph_id, step, em_steps, explanation.candidate.payload.spacegroup, explanation.candidate.source, explanation.residual_now, explanation.residual_em, use_cf, cfg24.region.relative_slack_now, cfg24.region.relative_slack_em, cfg24.region.anchor_tiebreak_delta, cfg24.selection.lambda_em, cfg24.selection.lambda_shift, cfg24.region.lambdas, cfg24.region.perturb_sigmas, ALGO24_USE_CF), _runner)


def eval_region_candidates(case: GraphCase, region_candidates):
    rows = []
    for cand in region_candidates:
        ev = evaluate_frac(case, cand.frac_coords_model_order)
        rows.append({
            'candidate_id': cand.candidate_id,
            'source': cand.source,
            'lambda_value': safe_metric_float(cand.lambda_value),
            'sigma_q': safe_metric_float(cand.sigma_q),
            'radial_index': int(cand.radial_index),
            'accepted_by_region': bool(cand.accepted_by_region),
            'now_cost': safe_metric_float(cand.now_cost),
            'em_cost': safe_metric_float(cand.em_cost),
            'now_cost_cap': safe_metric_float(cand.now_cost_cap),
            'em_cost_cap': safe_metric_float(cand.em_cost_cap),
            'shift_cost': safe_metric_float(cand.shift_cost),
            'anchor_score': safe_metric_float(cand.anchor_score),
            'cf_nll': safe_metric_float(cand.cf_nll),
            'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
            'rmse': safe_metric_float(ev.get('rmse')),
            'match': safe_metric_bool(ev.get('match')),
            'valid': safe_metric_bool(ev.get('valid')),
        })
    return pd.DataFrame(rows)


def shortlist_region_candidates(region_candidates, *, max_candidates: int = 5):
    accepted = [cand for cand in region_candidates if cand.accepted_by_region]
    if not accepted:
        return tuple()
    accepted = sorted(accepted, key=lambda cand: (float(cand.anchor_score), float(cand.now_cost), float(cand.em_cost), float(cand.sigma_q)))
    picks = list(accepted[: max(1, int(max_candidates))])
    q_now = next((cand for cand in accepted if abs(float(cand.lambda_value)) <= 1.0e-12 and abs(float(cand.sigma_q)) <= 1.0e-12), None)
    q_em = next((cand for cand in accepted if abs(float(cand.lambda_value) - 1.0) <= 1.0e-12 and abs(float(cand.sigma_q)) <= 1.0e-12), None)
    if q_now is not None:
        picks.append(q_now)
    if q_em is not None:
        picks.append(q_em)
    unique = []
    seen = set()
    for cand in picks:
        if cand is None or cand.candidate_id in seen:
            continue
        seen.add(cand.candidate_id)
        unique.append(cand)
    return tuple(unique)


def run_algo24_survival(case: GraphCase, step: int, em_steps: int, region_candidates, *, alpha: float, cfg24=ALGO24_CFG):
    state, f0_hat = get_state_and_clean(case, step)
    return algorithm24_select_by_post_update_survival(
        model=runner.model,
        state=state,
        f0_anchor=f0_hat,
        region_candidates=tuple(c for c in region_candidates if c.accepted_by_region),
        alpha=float(alpha),
        beta=1.0,
        config=cfg24,
    )


def _algo_state_from_sampling(prepared, *, t_graph, t_nodes):
    return make_algo_state_from_raw(
        f=prepared['f_t'],
        v=prepared['v_t'],
        l=prepared['l_t'],
        atom_types=prepared['a_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
        t_graph=t_graph,
        t_nodes=t_nodes,
    )


def _clone_prepared_sampling_state(prepared: dict[str, Any]) -> dict[str, Any]:
    out = dict(prepared)
    for key, value in list(prepared.items()):
        if torch.is_tensor(value):
            out[key] = value.detach().clone()
    return out


def _pc_facit_step(prepared: dict[str, Any], times, *, tau: float = 0.25, n_correction_steps: int = 1):
    for _ in range(max(int(n_correction_steps), 1)):
        preds_corr = runner.model.score_network(
            t=times.now.graph,
            pos=prepared['f_t'],
            v=prepared['v_t'],
            h=prepared['a_t'],
            l=prepared['l_t'],
            node_index=prepared['node_index'],
            edge_node_index=prepared['edge_node_index'],
        )
        prepared['f_t'], prepared['v_t'] = prepared['sampling_tdm'].reverse_step_corrector(
            t=times.now.nodes,
            f_t=prepared['f_t'],
            v_t=prepared['v_t'],
            pred_v=preds_corr['v'],
            dt=times.dt,
            index=prepared['node_index'],
            tau=float(tau),
        )
        prepared['l_t'] = prepared['sampling_diffusion_l'].reverse_step_corrector(
            t=times.now.lattice,
            x_t=prepared['l_t'],
            pred=preds_corr['l'],
            tau=float(tau),
        )

    preds_pred = runner.model.score_network(
        t=times.now.graph,
        pos=prepared['f_t'],
        v=prepared['v_t'],
        h=prepared['a_t'],
        l=prepared['l_t'],
        node_index=prepared['node_index'],
        edge_node_index=prepared['edge_node_index'],
    )
    prepared['f_t'], prepared['v_t'] = prepared['sampling_tdm'].reverse_step_predictor(
        t=times.now.nodes,
        f_t=prepared['f_t'],
        v_t=prepared['v_t'],
        pred_v=preds_pred['v'],
        index=prepared['node_index'],
        dt=times.dt,
    )
    prepared['l_t'] = prepared['sampling_diffusion_l'].reverse_step_predictor(
        t=times.now.lattice,
        x_t=prepared['l_t'],
        pred=preds_pred['l'],
        dt=times.dt,
    )


def run_pc_trajectory_cached(case: GraphCase, *, seed: int, n_steps: int = 800, checkpoints=(600, 650, 700, 750, 800), tau: float = 0.25, n_correction_steps: int = 1):
    def _runner():
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        set_seed(int(seed) + 810000 * int(case.graph_id) + int(n_steps))
        prepared = runner.model._prepare_csp_sampling(
            batch=batch_view,
            n_steps=int(n_steps),
            t_start=1.0,
            t_final=1.0e-6,
        )
        checkpoint_map = {}
        grid = prepared['sampling_time_grid']
        times_list = list(iter_sampling_times(batch=prepared['batch'], grid=grid))
        checkpoint_set = {int(v) for v in checkpoints}
        with torch.no_grad():
            for times in times_list:
                _pc_facit_step(prepared, times, tau=float(tau), n_correction_steps=int(n_correction_steps))
                completed_step = int(times.step) + 1
                if completed_step in checkpoint_set:
                    checkpoint_map[int(completed_step)] = {
                        'prepared': _clone_prepared_sampling_state(prepared),
                        'time_index': int(completed_step),
                    }
        final_eval = evaluate_arrays(case, prepared['f_t'], prepared['l_t'].reshape(-1), prepared['a_t'])
        return {
            'checkpoints': checkpoint_map,
            'final_f': prepared['f_t'].detach().clone(),
            'final_v': prepared['v_t'].detach().clone(),
            'final_l': prepared['l_t'].detach().clone().reshape(-1),
            'final_a': prepared['a_t'].detach().clone(),
            'final_eval': final_eval,
        }
    return _cached(_cache_key('pc_trajectory', case.graph_id, seed, n_steps, tuple(checkpoints), tau, n_correction_steps), _runner)


def run_baseline_pc800(case: GraphCase, *, seed: int, n_steps: int = 800):
    cache = run_pc_trajectory_cached(case, seed=int(seed), n_steps=int(n_steps))
    return {
        'f': cache['final_f'],
        'v': cache['final_v'],
        'l': cache['final_l'],
        'a': cache['final_a'],
        'eval': cache['final_eval'],
        'interventions': [],
    }


def run_algorithm24_anchor_sampler(case: GraphCase, *, seed: int, n_steps: int = 800, start_step: int = 600, step_interval: int = 50, em_steps: int = 300, alpha: float = 0.03, max_q: int = 5, cfg24=ALGO24_CFG):
    def _runner():
        pc_cache = run_pc_trajectory_cached(case, seed=int(seed), n_steps=int(n_steps))
        checkpoint = pc_cache['checkpoints'][int(start_step)]
        prepared = _clone_prepared_sampling_state(checkpoint['prepared'])
        batch_view = make_single_graph_batch_view(case, ref_tensor=case.gt_frac)
        times_list = list(iter_sampling_times(batch=prepared['batch'], grid=prepared['sampling_time_grid']))
        intervention_rows = []
        checkpoint_steps = set(range(int(start_step), int(n_steps), int(step_interval)))
        with torch.no_grad():
            for times in times_list[int(start_step):]:
                current_step = int(times.step)
                if current_step in checkpoint_steps:
                    algo_state = make_algo_state_from_raw(
                        f=prepared['f_t'],
                        v=prepared['v_t'],
                        l=prepared['l_t'],
                        atom_types=prepared['a_t'],
                        node_index=prepared['node_index'],
                        edge_node_index=prepared['edge_node_index'],
                        t_graph=times.now.graph,
                        t_nodes=times.now.nodes,
                    )
                    f0_hat = clean_fractional_estimate_22(state=algo_state, model=runner.model, config=ALGO22_CFG)
                    ranked = algorithm24_generate_template_candidates(
                        f0_hat=f0_hat,
                        state=algo_state,
                        space_group=int(case.requested_sg),
                        lattice_transform=runner.lattice_transform,
                        formula=cf_formula_for_case(case),
                        cf_likelihood=None,
                        config=cfg24,
                        source='pyxtal_only',
                        debug_label=f'g{case.graph_id}_algo24_sampler_s{current_step}',
                    )
                    ranked = tuple(ranked[:1])
                    if ranked:
                        lookahead = algorithm24_run_facit_em_lookahead(
                            model=runner.model,
                            batch=batch_view,
                            state=algo_state,
                            steps=int(em_steps),
                            t_final=float(cfg24.em.t_final),
                        )
                        explanations = algorithm24_explain_templates(
                            candidates=ranked,
                            target_now=f0_hat,
                            target_em=lookahead.f0_em,
                            target_atomic_numbers=algo_state.atom_types,
                            config=cfg24,
                        )
                        if explanations:
                            explanation = next((exp for exp in explanations if exp.passed_em_gate), explanations[0])
                            region = algorithm24_build_q_region(
                                explanation=explanation,
                                target_now=f0_hat,
                                target_em=lookahead.f0_em,
                                target_atomic_numbers=algo_state.atom_types,
                                lattice_feature=algo_state.l,
                                cf_energy_fn=None,
                                cf_formula=cf_formula_for_case(case),
                                config=cfg24,
                            )
                            shortlist = shortlist_region_candidates(region, max_candidates=max_q)
                            selected = select_by_anchor_then_cf(
                                region_candidates=shortlist,
                                anchor_delta=0.0,
                                use_cf_tiebreak=False,
                            ) if shortlist else None
                            if selected is not None:
                                update = algo24_backend.algorithm24_state_update(
                                    model=runner.model,
                                    state=algo_state,
                                    f0_anchor=f0_hat,
                                    frac_target=selected.frac_coords_model_order,
                                    alpha=float(alpha),
                                    beta=1.0,
                                    config=cfg24,
                                )
                                prepared['f_t'] = update.state.f.detach().clone()
                                prepared['v_t'] = update.state.v.detach().clone()
                                prepared['l_t'] = update.state.l.reshape_as(prepared['l_t']).detach().clone()
                                prepared['a_t'] = update.state.atom_types.detach().clone()
                                intervention_rows.append({
                                    'step': int(current_step),
                                    'num_shortlist': int(len(shortlist)),
                                    'selected_candidate_id': str(selected.candidate_id),
                                    'selected_lambda': safe_metric_float(selected.lambda_value),
                                    'selected_sigma': safe_metric_float(selected.sigma_q),
                                    'selected_anchor_score': safe_metric_float(selected.anchor_score),
                                })
                _pc_facit_step(prepared, times, tau=0.25, n_correction_steps=1)
        ev = evaluate_arrays(case, prepared['f_t'], prepared['l_t'].reshape(-1), prepared['a_t'])
        return {
            'f': prepared['f_t'].detach().clone(),
            'v': prepared['v_t'].detach().clone(),
            'l': prepared['l_t'].detach().clone().reshape(-1),
            'a': prepared['a_t'].detach().clone(),
            'eval': ev,
            'interventions': intervention_rows,
        }
    return _cached(_cache_key('algo24_anchor_sampler', case.graph_id, seed, n_steps, start_step, step_interval, em_steps, alpha, max_q, cfg24.region.relative_slack_now, cfg24.region.relative_slack_em), _runner)


## Test Group A — Foundation

Decision goal: verify that the oracle q/frame path is correct before trusting EM, CrystalFormer, or CPS conclusions.


In [9]:
rows_a = []
for case in GRAPH_CASES[:1]:
    payload = build_oracle_payload(case)
    template, q_gt = recover_oracle_template_from_payload(case, payload)
    state, f0_hat = get_state_and_clean(case, 700)
    lattice_matrix = decode_state_cell_matrix_22(state=state, lattice_transform=runner.lattice_transform)
    oracle_cand = materialize_candidate_from_template_22(
        template=template,
        q=q_gt.to(device=f0_hat.device, dtype=f0_hat.dtype),
        state=state,
        lattice_matrix=lattice_matrix,
        lattice_feature=state.l,
        source='oracle_q_identity',
        metadata={'mode': 'oracle_q_identity'},
        spacegroup=int(payload.spacegroup),
    )
    oracle_ev = evaluate_frac(case, oracle_cand.frac_coords_model_order)
    shift = torch.tensor([0.173], device=f0_hat.device, dtype=f0_hat.dtype)
    shifted_target = wrap01(case.gt_frac + shift)
    shifted_fit = fit_candidate_to_clean(case, state, shifted_target, oracle_cand, ALGO22_CFG)
    shifted_ev = evaluate_arrays(case, shifted_fit.frac_coords_model_order, case.gt_l_feature, case.atomic_numbers)
    perm = torch.arange(case.atomic_numbers.shape[0], device=case.atomic_numbers.device)
    first_species = int(case.atomic_numbers[0].item())
    species_idx = torch.where(case.atomic_numbers == first_species)[0]
    if len(species_idx) > 1:
        perm[species_idx] = species_idx.flip(0)
    perm_target = case.gt_frac[perm]
    perm_fit = fit_candidate_to_clean(case, state, perm_target, oracle_cand, ALGO22_CFG)
    rows_a.append({
        'test': 'algorithm24_a1_a2_foundation',
        'graph': case.graph_id,
        'oracle_frac_rmse': safe_metric_float(oracle_ev.get('frac_rmse')),
        'oracle_rmse': safe_metric_float(oracle_ev.get('rmse')),
        'oracle_match': safe_metric_bool(oracle_ev.get('match')),
        'oracle_valid': safe_metric_bool(oracle_ev.get('valid')),
        'shifted_fit_frac_rmse': safe_metric_float(shifted_ev.get('frac_rmse')),
        'perm_fit_witness': safe_metric_float(perm_fit.witness),
        'PASS': bool(safe_metric_bool(oracle_ev.get('match')) and safe_metric_bool(oracle_ev.get('valid')) and safe_metric_float(oracle_ev.get('rmse')) < 1e-4),
        'status': 'INFO',
    })
safe_display_sorted(dataframe_result('algorithm24_a1_a2_foundation', rows_a), ['graph'])


/workspace/src/kldmPlus/sample_evaluation/sample_evaluation.py:482: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:206.)
  return torch.as_tensor(


,test,graph,oracle_frac_rmse,oracle_rmse,oracle_match,oracle_valid,shifted_fit_frac_rmse,perm_fit_witness,PASS,status
0,algorithm24_a1_a2_foundation,2,0.207584,1.130071e-07,True,True,0.170533,0.00033,True,INFO


## Test Group B — EM Explainability and Region Feasibility

Decision goal: check whether the EM endpoint is explainable under oracle `G`, and whether the `q_now \leftrightarrow q_EM` region contains candidates better than `q_now`.


In [10]:
rows_b1 = []
rows_b2 = []
rows_b3 = []
for case in GRAPH_CASES[:1]:
    payload = build_oracle_payload(case)
    chart = algo22_backend._get_wyckoff_dof_chart(payload)
    q_gt = torch.as_tensor(chart.q_ref, device=case.gt_frac.device, dtype=case.gt_frac.dtype).reshape(-1)
    for step in [500]:
        for em_steps in [300]:
            explain = explain_case_step(case, step, em_steps, top_k=1)
            lookahead = explain['lookahead']
            em_ev = evaluate_arrays(case, lookahead.f0_em, lookahead.l0_em, lookahead.a0_em)
            explanations = list(explain['explanations'])
            oracle_rank = -1
            for idx, exp in enumerate(sorted(explanations, key=lambda x: (x.residual_em, x.cross_em_to_now)), start=1):
                if exp.candidate.template is not None and template_signature(exp.candidate.template) == oracle_signature(case):
                    oracle_rank = idx
                    break
            best_exp = min(explanations, key=lambda x: (x.residual_em, x.cross_em_to_now)) if explanations else None
            rows_b1.append({
                'test': 'algorithm24_b1_em_explainability',
                'graph': case.graph_id,
                'step': int(step),
                'em_steps': int(em_steps),
                'raw_EM_frac_rmse_eval_only': safe_metric_float(em_ev.get('frac_rmse')),
                'raw_EM_rmse_eval_only': safe_metric_float(em_ev.get('rmse')),
                'raw_EM_match_eval_only': safe_metric_bool(em_ev.get('match')),
                'raw_EM_valid_eval_only': safe_metric_bool(em_ev.get('valid')),
                'best_EM_to_oracleG_residual': safe_metric_float(best_exp.residual_em) if best_exp is not None else float('nan'),
                'best_cross_em_to_now': safe_metric_float(best_exp.cross_em_to_now) if best_exp is not None else float('nan'),
                'oracle_template_rank_by_EM_residual_eval_only': int(oracle_rank),
                'PASS': bool(best_exp is not None and best_exp.residual_em <= ALGO24_CFG.region.em_residual_gate),
                'status': 'INFO',
            })
            for exp in explanations:
                q_now_dist = float(torch.sqrt(cf_backend.torus_mse(exp.q_now.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if exp.q_now.numel() == q_gt.numel() else float('nan')
                q_em_dist = float(torch.sqrt(cf_backend.torus_mse(exp.q_em.reshape(-1), q_gt.reshape(-1)).clamp_min(0.0)).detach().item()) if exp.q_em.numel() == q_gt.numel() else float('nan')
                now_ev = evaluate_frac(case, exp.frac_now)
                emq_ev = evaluate_frac(case, exp.frac_em)
                rows_b2.append({
                    'test': 'algorithm24_b2_q_now_vs_q_em',
                    'graph': case.graph_id,
                    'step': int(step),
                    'em_steps': int(em_steps),
                    'template_matches_GT_eval_only': bool(exp.candidate.template is not None and template_signature(exp.candidate.template) == oracle_signature(case)),
                    'residual_now': safe_metric_float(exp.residual_now),
                    'residual_em': safe_metric_float(exp.residual_em),
                    'cross_em_to_now': safe_metric_float(exp.cross_em_to_now),
                    'q_now_distance_to_GT': safe_metric_float(q_now_dist),
                    'q_em_distance_to_GT': safe_metric_float(q_em_dist),
                    'q_now_frac_rmse': safe_metric_float(now_ev.get('frac_rmse')),
                    'q_em_frac_rmse': safe_metric_float(emq_ev.get('frac_rmse')),
                    'q_em_beats_q_now': strict_improves(emq_ev.get('frac_rmse'), now_ev.get('frac_rmse')),
                    'passed_em_gate': bool(exp.passed_em_gate),
                    'PASS': True,
                    'status': 'INFO',
                })
                region = build_region(case, step, em_steps, exp, use_cf=False)
                region_df = eval_region_candidates(case, region)
                accepted = region_df[region_df['accepted_by_region'] == True].copy()
                if len(accepted):
                    interp = accepted[(accepted['lambda_value'] >= 0.0) & (accepted['lambda_value'] <= 1.0)]
                    extra = accepted[(accepted['lambda_value'] < 0.0) | (accepted['lambda_value'] > 1.0)]
                    best_region_frac = finite_min(accepted['frac_rmse'])
                    best_region_rmse = finite_min(accepted['rmse'])
                    best_interp_frac = finite_min(interp['frac_rmse']) if len(interp) else float('nan')
                    best_extra_frac = finite_min(extra['frac_rmse']) if len(extra) else float('nan')
                    best_row = accepted.sort_values(['frac_rmse', 'rmse'], kind='mergesort').iloc[0]
                else:
                    best_region_frac = float('nan')
                    best_region_rmse = float('nan')
                    best_interp_frac = float('nan')
                    best_extra_frac = float('nan')
                    best_row = None
                rows_b3.append({
                    'test': 'algorithm24_b3_region_contains_better_q',
                    'graph': case.graph_id,
                    'step': int(step),
                    'em_steps': int(em_steps),
                    'template_matches_GT_eval_only': bool(exp.candidate.template is not None and template_signature(exp.candidate.template) == oracle_signature(case)),
                    'region_num_candidates': int(len(region_df)),
                    'region_num_accepted': int(len(accepted)),
                    'q_now_frac_rmse': safe_metric_float(now_ev.get('frac_rmse')),
                    'q_em_frac_rmse': safe_metric_float(emq_ev.get('frac_rmse')),
                    'best_interpolation_frac_rmse': safe_metric_float(best_interp_frac),
                    'best_extrapolation_frac_rmse': safe_metric_float(best_extra_frac),
                    'best_tube_frac_rmse': safe_metric_float(best_region_frac),
                    'best_tube_rmse': safe_metric_float(best_region_rmse),
                    'best_lambda': safe_metric_float(best_row['lambda_value']) if best_row is not None else float('nan'),
                    'best_sigma': safe_metric_float(best_row['sigma_q']) if best_row is not None else float('nan'),
                    'region_contains_better_than_q_now': strict_improves(best_region_frac, now_ev.get('frac_rmse')) or strict_improves(best_region_rmse, now_ev.get('rmse')),
                    'PASS': bool(len(accepted) > 0),
                    'status': 'INFO',
                })
safe_display_sorted(dataframe_result('algorithm24_b1_em_explainability', rows_b1), ['graph', 'step', 'em_steps'])
safe_display_sorted(dataframe_result('algorithm24_b2_q_now_vs_q_em', rows_b2), ['graph', 'step', 'em_steps', 'template_matches_GT_eval_only'])
safe_display_sorted(dataframe_result('algorithm24_b3_region_contains_better_q', rows_b3), ['graph', 'step', 'em_steps', 'template_matches_GT_eval_only'])


KeyboardInterrupt: 

## Test Group C — CrystalFormer Local Ranking in the Region

Decision goal: inside the already filtered EM/KLDM-compatible q-region, determine whether CrystalFormer NLL carries useful local ranking signal.


In [ ]:
rows_c = []
rows_c_gate = []
for case in GRAPH_CASES[:1]:
    for step in [600]:
        for em_steps in [300]:
            base_explain = explain_case_step(case, step, em_steps, top_k=1)
            exp = base_explain['explanations'][0]
            for rho in ALGO24_GATE_RHOS:
                cfg_gate = make_algo24_cfg(rho_now=rho, rho_em=rho)
                region = build_region(case, step, em_steps, exp, use_cf=False, cfg24=cfg_gate)
                region_df = eval_region_candidates(case, region)
                accepted = region_df[region_df['accepted_by_region'] == True].copy()
                best_region = accepted.sort_values(['frac_rmse', 'rmse'], kind='mergesort').iloc[0] if len(accepted) else None
                by_anchor = accepted.sort_values(['anchor_score', 'frac_rmse'], kind='mergesort').iloc[0] if len(accepted) else None
                rows_c_gate.append({
                    'test': 'algorithm24_c2_gate_tightness',
                    'graph': case.graph_id,
                    'step': int(step),
                    'em_steps': int(em_steps),
                    'rho_now': float(rho),
                    'rho_em': float(rho),
                    'num_candidates': int(len(region_df)),
                    'num_accepted': int(len(accepted)),
                    'oracle_best_accepted_frac_rmse': safe_metric_float(best_region['frac_rmse']) if best_region is not None else float('nan'),
                    'anchor_selected_frac_rmse': safe_metric_float(by_anchor['frac_rmse']) if by_anchor is not None else float('nan'),
                    'did_gate_drop_oracle_best': bool(len(region_df) > 0 and len(accepted) == 0),
                    'PASS': bool(len(accepted) > 0),
                    'status': 'INFO',
                })
            region = build_region(case, step, em_steps, exp, use_cf=False)
            region_df = eval_region_candidates(case, region)
            accepted = region_df[region_df['accepted_by_region'] == True].copy()
            if len(accepted):
                accepted['anchor_rank'] = accepted['anchor_score'].rank(method='average')
                accepted['rmse_rank'] = accepted['frac_rmse'].rank(method='average')
                spearman_anchor = float(accepted['anchor_rank'].corr(accepted['rmse_rank'], method='pearson')) if len(accepted) > 1 else float('nan')
                by_anchor = accepted.sort_values(['anchor_score', 'frac_rmse'], kind='mergesort').iloc[0]
                best_region = accepted.sort_values(['frac_rmse', 'rmse'], kind='mergesort').iloc[0]
            else:
                spearman_anchor = float('nan')
                by_anchor = best_region = None
            rows_c.append({
                'test': 'algorithm24_c1_anchor_local_ranking',
                'graph': case.graph_id,
                'step': int(step),
                'em_steps': int(em_steps),
                'template_matches_GT_eval_only': bool(exp.candidate.template is not None and template_signature(exp.candidate.template) == oracle_signature(case)),
                'num_region_candidates': int(len(region_df)),
                'num_region_accepted': int(len(accepted)),
                'spearman_anchor_vs_gt_frac_rmse': safe_metric_float(spearman_anchor),
                'top1_by_anchor_frac_rmse': safe_metric_float(by_anchor['frac_rmse']) if by_anchor is not None else float('nan'),
                'top1_by_anchor_rmse': safe_metric_float(by_anchor['rmse']) if by_anchor is not None else float('nan'),
                'oracle_best_region_frac_rmse': safe_metric_float(best_region['frac_rmse']) if best_region is not None else float('nan'),
                'best_anchor_lambda': safe_metric_float(by_anchor['lambda_value']) if by_anchor is not None else float('nan'),
                'best_anchor_sigma': safe_metric_float(by_anchor['sigma_q']) if by_anchor is not None else float('nan'),
                'anchor_hits_oracle_best': bool(by_anchor is not None and best_region is not None and abs(float(by_anchor['frac_rmse']) - float(best_region['frac_rmse'])) <= ALGO24_STRICT_TOL),
                'PASS': bool(len(accepted) > 0),
                'status': 'INFO',
            })
safe_display_sorted(dataframe_result('algorithm24_c1_anchor_local_ranking', rows_c), ['graph', 'step', 'em_steps', 'template_matches_GT_eval_only'])
safe_display_sorted(dataframe_result('algorithm24_c2_gate_tightness', rows_c_gate), ['graph', 'step', 'em_steps', 'rho_now'])


## Test Group D — One-Step CPS Survival and Safety Gate

Decision goal: check whether candidates from the EM-guided region survive a weak CPS intervention better than the simpler `q_now` baseline, and whether EM disagreement can be used as a safety gate.


In [ ]:
rows_d1 = []
rows_d2 = []
for case in GRAPH_CASES[:1]:
    for step in [600]:
        for em_steps in [300]:
            explain = explain_case_step(case, step, em_steps, top_k=1)
            for exp in explain['explanations']:
                region = build_region(case, step, em_steps, exp, use_cf=False)
                shortlist = shortlist_region_candidates(region, max_candidates=3)
                if not shortlist:
                    continue
                by_anchor = select_by_anchor_then_cf(region_candidates=shortlist, anchor_delta=0.0, use_cf_tiebreak=False)
                q_now_baseline = next((cand for cand in shortlist if abs(float(cand.lambda_value)) <= 1e-12 and abs(float(cand.sigma_q)) <= 1e-12), shortlist[0])
                q_em_axis = next((cand for cand in shortlist if abs(float(cand.lambda_value) - 1.0) <= 1e-12 and abs(float(cand.sigma_q)) <= 1e-12), q_now_baseline)
                for alpha in ALGO24_ALPHA_VALUES:
                    selection = run_algo24_survival(case, step, em_steps, shortlist, alpha=alpha)
                    candidate_map = {ev.region_candidate.candidate_id: ev for ev in selection.evaluations}
                    candidates_to_report = [
                        ('q_now_baseline', q_now_baseline, candidate_map.get(q_now_baseline.candidate_id)),
                        ('q_em_axis', q_em_axis, candidate_map.get(q_em_axis.candidate_id)),
                        ('q_best_anchor', by_anchor, candidate_map.get(by_anchor.candidate_id) if by_anchor is not None else None),
                        ('q_best_post_update', selection.best.region_candidate, selection.best),
                    ]
                    seen = set()
                    for label, cand_like, eval_like in candidates_to_report:
                        if cand_like is None or eval_like is None or label in seen:
                            continue
                        seen.add(label)
                        after_ev = evaluate_frac(case, eval_like.f0_after)
                        rows_d1.append({
                            'test': 'algorithm24_d1_shortlist_survival',
                            'graph': case.graph_id,
                            'step': int(step),
                            'em_steps': int(em_steps),
                            'alpha': float(alpha),
                            'selector': label,
                            'shortlist_size': int(len(shortlist)),
                            'candidate_id': cand_like.candidate_id,
                            'candidate_lambda': safe_metric_float(cand_like.lambda_value),
                            'candidate_sigma': safe_metric_float(cand_like.sigma_q),
                            'candidate_now_cost': safe_metric_float(cand_like.now_cost),
                            'candidate_em_cost': safe_metric_float(cand_like.em_cost),
                            'candidate_anchor_score': safe_metric_float(cand_like.anchor_score),
                            'projected_frac_rmse_eval_only': safe_metric_float(evaluate_frac(case, eval_like.update.f0_projected).get('frac_rmse')),
                            'post_geometry_cost': safe_metric_float(eval_like.post_geometry_cost),
                            'post_frac_rmse_eval_only': safe_metric_float(after_ev.get('frac_rmse')),
                            'post_rmse_eval_only': safe_metric_float(after_ev.get('rmse')),
                            'post_match_eval_only': safe_metric_bool(after_ev.get('match')),
                            'post_valid_eval_only': safe_metric_bool(after_ev.get('valid')),
                            'shift_distance': safe_metric_float(eval_like.shift_distance),
                            'PASS': True,
                            'status': 'INFO',
                        })
                U = safe_metric_float(algo24_backend.algorithm24_torus_mse(exp.frac_now, exp.frac_em))
                for threshold in ALGO24_SAFETY_THRESHOLDS:
                    rows_d2.append({
                        'test': 'algorithm24_d2_em_agreement_safety_gate',
                        'graph': case.graph_id,
                        'step': int(step),
                        'em_steps': int(em_steps),
                        'agreement_U': U,
                        'threshold': float(threshold),
                        'project_if_small_U': bool(np.isfinite(U) and U <= float(threshold)),
                        'passed_em_gate': bool(exp.passed_em_gate),
                        'status': 'INFO',
                        'PASS': True,
                    })
safe_display_sorted(dataframe_result('algorithm24_d1_shortlist_survival', rows_d1), ['graph', 'step', 'em_steps', 'alpha', 'selector'])
safe_display_sorted(dataframe_result('algorithm24_d2_em_agreement_safety_gate', rows_d2), ['graph', 'step', 'em_steps', 'threshold'])


## Test Group E — Lightweight Verdict

Decision goal: summarize whether Algorithm24 currently looks like a promising main method, a useful safety gate, or something to drop.


In [ ]:
rows_e1 = []
rows_e2 = []
for case in GRAPH_CASES[:1]:
    for seed in ALGO24_SAMPLER_SEEDS:
        baseline = run_baseline_pc800(case, seed=int(seed), n_steps=800)
        algo24_run = run_algorithm24_anchor_sampler(
            case,
            seed=int(seed),
            n_steps=800,
            start_step=600,
            step_interval=50,
            em_steps=100, #int(ALGO24_CFG.em.lookahead_steps),
            alpha=0.06,
            max_q=1,
        )
        for method, result in [
            ('baseline_pc800', baseline),
            ('algorithm24_anchor_tube_pc600to800_top5', algo24_run),
        ]:
            ev = result['eval']
            rows_e1.append({
                'test': 'algorithm24_e1_sampler_compare',
                'graph': case.graph_id,
                'seed': int(seed),
                'method': method,
                'frac_rmse': safe_metric_float(ev.get('frac_rmse')),
                'rmse': safe_metric_float(ev.get('rmse')),
                'match': safe_metric_bool(ev.get('match')),
                'valid': safe_metric_bool(ev.get('valid')),
                'num_interventions': int(len(result.get('interventions', []))),
                'PASS': True,
                'status': 'INFO',
            })
    sampler_df = pd.DataFrame([row for row in rows_e1 if row['graph'] == case.graph_id])
    baseline_rows = sampler_df[sampler_df['method'] == 'baseline_pc800']
    algo24_rows = sampler_df[sampler_df['method'] == 'algorithm24_anchor_tube_pc600to800_top5']
    baseline_best = finite_mean(baseline_rows['frac_rmse']) if len(baseline_rows) else float('nan')
    algo24_best = finite_mean(algo24_rows['frac_rmse']) if len(algo24_rows) else float('nan')
    sampler_beats_baseline = strict_improves(algo24_best, baseline_best)
    foundation = result_tables.get('algorithm24_a1_a2_foundation', pd.DataFrame())
    em_explain = result_tables.get('algorithm24_b1_em_explainability', pd.DataFrame())
    region = result_tables.get('algorithm24_b3_region_contains_better_q', pd.DataFrame())
    anchor_local = result_tables.get('algorithm24_c1_anchor_local_ranking', pd.DataFrame())
    survival = result_tables.get('algorithm24_d1_shortlist_survival', pd.DataFrame())
    foundation_ok = bool(len(foundation) and foundation['PASS'].all())
    em_ok = bool(len(em_explain) and em_explain['PASS'].any())
    region_ok = bool(len(region) and region['region_contains_better_than_q_now'].any())
    anchor_ok = bool(len(anchor_local) and anchor_local['anchor_hits_oracle_best'].any())
    anchor_survival = survival[survival['selector'] == 'q_best_anchor'] if len(survival) else pd.DataFrame()
    qnow_survival = survival[survival['selector'] == 'q_now_baseline'] if len(survival) else pd.DataFrame()
    anchor_survives = False
    if len(anchor_survival) and len(qnow_survival):
        merged = anchor_survival.merge(qnow_survival, on=['graph', 'step', 'em_steps', 'alpha'], suffixes=('_anchor', '_qnow'))
        anchor_survives = bool(len(merged) and np.any(merged['post_frac_rmse_eval_only_anchor'] <= merged['post_frac_rmse_eval_only_qnow'] + ALGO24_STRICT_TOL))
    if foundation_ok and em_ok and region_ok and anchor_ok and sampler_beats_baseline:
        recommendation = 'promising_algorithm24_anchor_tube_sampler'
    elif foundation_ok and em_ok and region_ok and anchor_ok:
        recommendation = 'local_positive_not_sampler_positive_yet'
    elif foundation_ok and em_ok and region_ok:
        recommendation = 'use_em_region_without_sampler_claim'
    else:
        recommendation = 'keep_algorithm22_simpler_default'
    rows_e2.append({
        'test': 'algorithm24_e2_verdict',
        'graph': case.graph_id,
        'foundation_ok': foundation_ok,
        'em_explainability_ok': em_ok,
        'region_contains_better_q': region_ok,
        'anchor_hits_oracle_best': anchor_ok,
        'anchor_survives_weak_cps': anchor_survives,
        'sampler_beats_baseline': sampler_beats_baseline,
        'recommendation': recommendation,
        'PASS': foundation_ok and em_ok,
        'status': 'INFO',
    })
safe_display_sorted(dataframe_result('algorithm24_e1_sampler_compare', rows_e1), ['graph', 'seed', 'method'])
safe_display_sorted(dataframe_result('algorithm24_e2_verdict', rows_e2), ['graph'])


KeyboardInterrupt: 

## Summary


In [ ]:
summary_rows = []
for key in [
    'algorithm24_a1_a2_foundation',
    'algorithm24_b1_em_explainability',
    'algorithm24_b2_q_now_vs_q_em',
    'algorithm24_b3_region_contains_better_q',
    'algorithm24_c1_anchor_local_ranking',
    'algorithm24_c2_gate_tightness',
    'algorithm24_d1_shortlist_survival',
    'algorithm24_d2_em_agreement_safety_gate',
    'algorithm24_e1_sampler_compare',
    'algorithm24_e2_verdict',
]:
    df = result_tables.get(key)
    if df is None or not len(df):
        summary_rows.append({'table': key, 'rows': 0, 'pass_rate': np.nan})
        continue
    pass_rate = float(np.mean(df['PASS'].astype(float))) if 'PASS' in df.columns else np.nan
    summary_rows.append({'table': key, 'rows': int(len(df)), 'pass_rate': pass_rate})
display(pd.DataFrame(summary_rows))
